**Path**

In [ ]:
from pathlib import Path

# Resolve paths relative to the repository root.
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
elif not (ROOT_DIR / "notebooks").exists() and (ROOT_DIR.parent / "notebooks").exists():
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data" / "raw"
SELECTED_FILES_CSV = ROOT_DIR / "data" / "selected_files.csv"
RESULTS_DIR = ROOT_DIR / "results"
MODELS_DIR = ROOT_DIR / "models"
PDF_DIR = RESULTS_DIR / "main_experiment"
SAVED_MODEL_DIR = MODELS_DIR / "main_experiment"

RUN_NAME = PDF_DIR.name
RUN_PLOT_DIR = PDF_DIR
ODE_PLOT_DIR = RUN_PLOT_DIR / "ODE_plot"
SMOOTHING_PLOT_DIR = RUN_PLOT_DIR / "smoothing"
EVAL_METRIC_TXT_PATH = RUN_PLOT_DIR / f"#Eval_Metric_{RUN_NAME}.txt"
FILE_LIST_TXT_PATH = RUN_PLOT_DIR / f"#File_List_{RUN_NAME}.txt"
EXPERIMENT_SETTING_TXT_PATH = RUN_PLOT_DIR / f"#Experiment_Setting_{RUN_NAME}.txt"
WINDOW_PARAMS_CSV_PATH = SAVED_MODEL_DIR / f"{RUN_NAME}_window_params_from_results.csv"
EXPERIMENT_CONFIG_JSON_PATH = SAVED_MODEL_DIR / "experiment_config.json"
PARAM_CSV_PATH = SAVED_MODEL_DIR / "pinn_parameters.csv"
MODEL_PARAMS_CSV_PATH = PARAM_CSV_PATH

for _path in [RUN_PLOT_DIR, ODE_PLOT_DIR, SMOOTHING_PLOT_DIR, SAVED_MODEL_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

BASE_DIR = DATA_DIR

SMOOTHING_METHOD = "moving_avg"
SMOOTHING_WIN_SIZE = 2500
SMOOTHING_POLYORDER = 3


**Library**

In [2]:
import os, glob, re
import numpy as np
import torch, torch.nn as nn
import soundfile as sf
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
#import sys
import csv
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from collections import Counter, defaultdict
from torch.quasirandom import SobolEngine
#from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import calinski_harabasz_score
from scipy.stats   import f_oneway   
from matplotlib.ticker import MaxNLocator
#from matplotlib import font_manager as fm
#from scipy.signal import find_peaks
from scipy.signal import savgol_filter
#from scipy.signal import welch
#from scipy import signal
import warnings
from scipy.ndimage import gaussian_filter1d
from scipy.integrate import solve_ivp
warnings.filterwarnings("ignore", category=DeprecationWarning)

**Global Seed**

In [3]:
SEED = 1111
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

**Global win_sec, overlap**

In [4]:
WIN_SEC = 0.1
MY_OVERLAP = 0.0

**PDF Saving and Font Configuration**

In [5]:
# Use matplotlib default bundled fonts instead of local font files.
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["mathtext.fontset"] = "dejavusans"
plt.rcParams["text.usetex"] = False

# Embed fonts in PDF outputs.
plt.rcParams["pdf.fonttype"] = 42

# Create the output directory for PDF figures.
pdf_dir = PDF_DIR
os.makedirs(pdf_dir, exist_ok=True)  

# Save the current figure as a PDF file.
def save_plot_as_pdf(pdf_filename):
    plt.savefig(pdf_filename, format="pdf", dpi=300, bbox_inches="tight")

**wav file load**

In [ ]:
@dataclass
class Record:
    idx: int
    t: np.ndarray
    x: np.ndarray
    fs: float
    group: str
    filename: str


def _find_selected_wav(data_dir: Path, filename: str) -> Path:
    matches = sorted(data_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} under {data_dir}. "
            "Download the Kaggle dataset and place the extracted files under data/raw/."
        )
    if len(matches) > 1:
        print(f"[WARN] Multiple matches for {filename}; using {matches[0]}")
    return matches[0]


def load_selected_files(data_dir=DATA_DIR, selected_csv=SELECTED_FILES_CSV):
    data_dir = Path(data_dir)
    selected_csv = Path(selected_csv)

    if not selected_csv.exists():
        raise FileNotFoundError(f"Selected file list not found: {selected_csv}")
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Raw data directory not found: {data_dir}. "
            "Download the Kaggle dataset and place the extracted files under data/raw/."
        )

    selected_df = pd.read_csv(selected_csv)
    required_cols = {"group", "filename"}
    missing_cols = required_cols - set(selected_df.columns)
    if missing_cols:
        raise ValueError(f"selected_files.csv is missing columns: {sorted(missing_cols)}")

    records_by_group = {"health": [], "wheeze": [], "crackle": []}
    for row in selected_df.itertuples(index=False):
        group = str(row.group).strip().lower()
        filename = str(row.filename).strip()
        if group not in records_by_group:
            raise ValueError(f"Unsupported group in selected_files.csv: {group}")

        wav_path = _find_selected_wav(data_dir, filename)
        x, fs = sf.read(wav_path, dtype="float32")
        if x.ndim == 2:
            x = x.mean(axis=1)

        t = np.arange(len(x), dtype=np.float32) / fs
        idx = len(records_by_group[group]) + 1

        print(f"[{group} #{idx}] {filename:45s} -> {len(x):,} samples ({len(x) / fs:.1f} s)")
        records_by_group[group].append(Record(
            idx=idx,
            t=t,
            x=x,
            fs=fs,
            group=group,
            filename=filename,
        ))

    return records_by_group


selected_records = load_selected_files()
file_H = selected_records["health"]
file_W = selected_records["wheeze"]
file_C = selected_records["crackle"]


RuntimeError: No wav files found for health in /home/namhy/ML/github/pinn-respiratory-sound/data/raw/health_wav

**Plot: raw data smoothing**

In [ ]:
def smooth_raw_data(file_list, method='moving_avg', win_size=101, polyorder=3, pdf_dir=None, return_data=False):
    """
    file_list : List returned by concat_group
                [(idx, t, x, fs, group), ...]

    method    : Smoothing method ('moving_avg' or 'savgol')

    win_size  : Size of the smoothing window

    polyorder : Polynomial order for the Savitzky-Golay filter
                (used only when method='savgol')
    """
    smoothed_list = []
    
    for record in file_list:
        idx = record.idx
        t = record.t
        x = record.x
        fs = record.fs
        group = record.group
        filename = record.filename
        if method == 'moving_avg':
            # moving average
            win = np.ones(win_size) / win_size
            x_smooth = np.convolve(x, win, mode='same')
        elif method == 'savgol':
            # Savitzky–Golay filter
            # win_size must be odd.
            if win_size % 2 == 0:
                win_size += 1
            x_smooth = savgol_filter(x, window_length=win_size, polyorder=polyorder)
        elif method == 'gaussian':
            sigma = win_size  
            x_smooth = gaussian_filter1d(x, sigma=sigma)
        else:
            raise ValueError("Unsupported method. Choose 'moving_avg' or 'savgol'.")

        # ---------- Parse the original prefix ID, recording ID, and segment information ----------
        file_id = None
        seg_num = None
        seg_range = None

        m_id = re.match(r"^(\d+)_", filename)
        if m_id:
            file_id = m_id.group(1) 

        m_seg = re.search(r"(seg\d+)_([0-9.]+-[0-9.]+)", filename)
        if m_seg:
            seg_num = m_seg.group(1)    
            seg_range = m_seg.group(2)  

        # Use a fallback strategy if parsing fails.
        if file_id is None:
            file_id = str(idx)

        if seg_num is None or seg_range is None:
            file_tag = os.path.splitext(filename)[0] 
        else:
            file_tag = f"file{file_id}_{seg_num}_{seg_range}"

        # Plot
        plt.figure(figsize=(16, 4))

        group_map = {'health': 'Healthy', 'wheeze': 'Wheeze', 'crackle': 'Crackle'}
        group_title = group_map.get(group.lower(), group)

        plt.plot(t, x, alpha=0.7, label='Raw')
        plt.plot(t, x_smooth, color='red', linewidth=1.0, label='Smoothed')

        duration_sec = len(x) / fs
        plt.title(f"Audio Waveform for {group_title} sound (Total Duration: {duration_sec:.0f} s)", fontsize=20)

        plt.xlabel("Time [s]", fontsize=24)
        plt.ylabel("Amplitude", fontsize=24)
        leg = plt.legend(fontsize=22, loc="upper right")
        leg.get_frame().set_alpha(0.4) 
        plt.xticks(fontsize=22)
        plt.yticks(fontsize=20)
        plt.ylim(-0.8, 0.8)
        plt.yticks(np.arange(-1.0, 1.1, 0.5))
        plt.gca().margins(x=0.025)
        plt.tight_layout()

        # save PDF
        if pdf_dir is None:
            pdf_dir = globals().get("SMOOTHING_PLOT_DIR", PDF_DIR / "smoothing")
        os.makedirs(pdf_dir, exist_ok=True)
        pdf_filename = os.path.join(pdf_dir, f"{group}_{idx}_win{win_size}.pdf")
        save_plot_as_pdf(pdf_filename)
        
        plt.show()
        
        if return_data:
            smoothed_list.append(Record(
                idx=idx,
                t=t,
                x=x_smooth,
                fs=fs,
                group=group,
                filename=filename
            ))
            
    if return_data:
        return smoothed_list

# call
# Moving average smoothing
sm_H = smooth_raw_data(file_H, method=SMOOTHING_METHOD, win_size=SMOOTHING_WIN_SIZE, polyorder=SMOOTHING_POLYORDER, return_data=True)
sm_W = smooth_raw_data(file_W, method=SMOOTHING_METHOD, win_size=SMOOTHING_WIN_SIZE, polyorder=SMOOTHING_POLYORDER, return_data=True)
sm_C = smooth_raw_data(file_C, method=SMOOTHING_METHOD, win_size=SMOOTHING_WIN_SIZE, polyorder=SMOOTHING_POLYORDER, return_data=True)

**Split window**

In [ ]:
def split_windows(t_cat, x_cat, fs_ref, win_sec, overlap=0.0,
                  target_pts=None, short_policy="drop"):
    """
    target_pts : Desired number of points for each window after processing
                (e.g., 441).
                - If None, keep the original number of samples.
                - If specified, resize the window using uniform resampling.

    short_policy : How to handle segments shorter than win_sec.
                - "drop": discard the segment (return no window)
                - "full": use the entire segment as a single window
                - "pad" : zero-pad to win_samples and use as a single window
    """
    win_samples = int(round(win_sec * fs_ref))
    stride_samples = int(round(win_sec * (1 - overlap) * fs_ref))  

    # --- Handle segments shorter than the target window length ---
    if len(x_cat) < win_samples:
        if short_policy == "drop":
            return np.empty((0, 0, 1), np.float32), np.empty((0, 0, 1), np.float32)
        elif short_policy == "full":
            t0 = (t_cat - t_cat[0]).astype(np.float32)
            x0 = x_cat.astype(np.float32)
            if target_pts is not None:
                idx = np.linspace(0, len(t0) - 1, num=min(target_pts, len(t0)), dtype=int)
                t0, x0 = t0[idx], x0[idx]
            return t0[None, :, None], x0[None, :, None]
        elif short_policy == "pad":
            pad_len = win_samples - len(x_cat)
            x_pad = np.pad(x_cat, (0, pad_len), mode="constant")
            t_pad = np.arange(len(x_pad), dtype=np.float32) / fs_ref
            t0 = (t_pad - t_pad[0]).astype(np.float32)
            x0 = x_pad.astype(np.float32)
            if target_pts is not None and target_pts < len(t0):
                idx = np.linspace(0, len(t0) - 1, num=target_pts, dtype=int)
                t0, x0 = t0[idx], x0[idx]
            return t0[None, :, None], x0[None, :, None]
        else:
            raise ValueError("short_policy must be one of ['drop','full','pad']")

    # --- Standard window segmentation ---
    starts = range(0, len(x_cat) - win_samples + 1, stride_samples)  
    t_windows, x_windows = [], []

    for s in starts:
        t0 = (t_cat[s:s+win_samples] - t_cat[s]).astype(np.float32)
        x0 = x_cat[s:s+win_samples].astype(np.float32)

        if target_pts is not None and target_pts < len(t0):
            idx = np.linspace(0, len(t0) - 1, num=target_pts, dtype=int)
            t0, x0 = t0[idx], x0[idx]

        t_windows.append(t0[:, None])
        x_windows.append(x0[:, None])

    return np.stack(t_windows, axis=0), np.stack(x_windows, axis=0)


# Split the smoothed signal into windows
window_H = [
    (*split_windows(rec.t, rec.x, rec.fs,
                    win_sec=WIN_SEC, overlap=MY_OVERLAP,
                    target_pts=441, short_policy="drop"),
     rec.fs, 'H', rec.filename)
    for rec in sm_H
]

window_W = [
    (*split_windows(rec.t, rec.x, rec.fs,
                    win_sec=WIN_SEC, overlap=MY_OVERLAP,
                    target_pts=441, short_policy="drop"),
     rec.fs, 'W', rec.filename)
    for rec in sm_W
]

window_C = [
    (*split_windows(rec.t, rec.x, rec.fs,
                    win_sec=WIN_SEC, overlap=MY_OVERLAP,
                    target_pts=441, short_policy="drop"),
     rec.fs, 'C', rec.filename)
    for rec in sm_C
]

print("\n[Healthy windows]")
for (t_win, x_win, fs, tag, filename) in window_H:
    print(f"{tag} | {filename:60s} | t_win={t_win.shape}  x_win={x_win.shape}")

print("\n[Wheeze windows]")
for (t_win, x_win, fs, tag, filename) in window_W:
    print(f"{tag} | {filename:60s} | t_win={t_win.shape}  x_win={x_win.shape}")

print("\n[Crackle windows]")
for (t_win, x_win, fs, tag, filename) in window_C:
    print(f"{tag} | {filename:60s} | t_win={t_win.shape}  x_win={x_win.shape}")


**Build PINN - *Duffing oscillator***

In [ ]:
class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)

class SirenLayer(nn.Module):
    """
    SIREN layer:
      - is_first=True 면 bound=1/n
      - else bound=√(6/n)/ω0
      - forward: sin(ω0 * (Wx + b))
    """
    def __init__(self, in_features, out_features, omega_0=1.0, is_first=False, bias=True):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            n = self.linear.in_features
            if self.is_first:
                bound = 1.0 / n
            else:
                bound = math.sqrt(6.0 / n) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)
            if self.linear.bias is not None:
                nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

# ── Fourier Encoder ─────────────────────────────────────────────
class FourierEncoder(nn.Module):
    def __init__(self, m: int = 16, sigma: float = 10.0):
        super().__init__()
        B = torch.randn(m, 1) * sigma
        self.register_buffer("B", B)         # Not trainable

    def forward(self, t: torch.Tensor) -> torch.Tensor:  # t: (N,1)
        proj = 2 * math.pi * t @ self.B.T                # (N,m)
        return torch.cat([proj.sin(), proj.cos()], dim=-1)  # (N,2m)

class MlpModel(nn.Module):
    def __init__(self,
                 in_dim,
                 n_hidden=3,
                 n_neurons=256,
                 activation='relu', # activation function selection
                 omega_0=1.0, # only used for siren
                 beta_init=None,
                 omega_init=None,  
                 B_init=None,  
                 fourier_m: int = 16,
                 fourier_sigma: float = 10.0, 
                 use_fourier: bool = False,
                 scaling_factor = 1.0):
        super().__init__()

        self.scaling_factor = scaling_factor

        # ────────────────────  Fourier encoder ───────────────────
        self.use_fourier = use_fourier
        if use_fourier:
            self.enc = FourierEncoder(fourier_m, fourier_sigma)
            in_dim += 2 * fourier_m   # expand the input dimension
        else:
            self.enc = None
         # --------------------------------------------------------

        self.act = activation.lower()
        self.omega_0 = omega_0

        layers = []
        # First hidden layer
        if self.act == 'siren':
            layers.append(SirenLayer(in_dim, n_neurons, omega_0=self.omega_0, is_first=True))
        else:
            layers.append(nn.Linear(in_dim, n_neurons))
            layers.append(self._get_activation())

        # Intermediate hidden layers
        for _ in range(n_hidden - 1):
            if self.act == 'siren':
                layers.append(SirenLayer(n_neurons, n_neurons, omega_0=self.omega_0, is_first=False))
            else:
                layers.append(nn.Linear(n_neurons, n_neurons))
                layers.append(self._get_activation())

        # Output later (linear)
        layers.append(nn.Linear(n_neurons, 1))

        self.net = nn.Sequential(*layers)

        # PINN parameters

        # Initialize alpha 
        alpha_init = 0.5
        self.log_alpha_hat = nn.Parameter(torch.log(torch.tensor([alpha_init/scaling_factor], dtype=torch.float32)))

        # Initialize beta
        self.log_beta_hat = nn.Parameter(torch.log(torch.tensor([beta_init/scaling_factor**2], dtype=torch.float32)))

        # Initialize omega 
        self.log_omega_hat = nn.Parameter(torch.log(torch.tensor([omega_init/scaling_factor], dtype=torch.float32)))

        # Initialize B
        self.log_B_hat = nn.Parameter(torch.log(torch.tensor([B_init/scaling_factor**2], dtype=torch.float32)))

        # Initialize gamma
        r_init = 1.0
        self.r_hat = nn.Parameter(torch.tensor([r_init/scaling_factor**2], dtype=torch.float32))

        # Initialize phi
        self.phi  = nn.Parameter(torch.zeros(1))

        # Weight initialization (non-SIREN)
        if self.act in ('relu', 'tanh', 'sine'):
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    if self.act == 'relu': # ReLU → He init
                        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                    else:
                        # Xavier initialization (Tanh and Sine) 
                        gain = (1.0 if self.act == 'sine' # sine → gain=1.0
                                else nn.init.calculate_gain('tanh')) # tanh → gain=calculate_gain('tanh')
                        nn.init.xavier_uniform_(m.weight, gain=gain)
                    nn.init.zeros_(m.bias)

    def _get_activation(self):
        if self.act == 'relu':
            return nn.ReLU()
        elif self.act == 'tanh':
            return nn.Tanh()
        elif self.act == 'sine':
            return Sine()
        else:
            raise ValueError(f"Unsupported activation: {self.act}")

    # ============== Enforce positivity ==============
    @property
    def alpha_hat(self):
        return torch.exp(self.log_alpha_hat)

    @property
    def beta_hat(self):
        return torch.exp(self.log_beta_hat)

    @property
    def omega_hat(self):
        return torch.exp(self.log_omega_hat)

    @property
    def B_hat(self):
        return torch.exp(self.log_B_hat)

    # ========== Convert back to physical units ==========
    @property
    def alpha(self):
        return self.alpha_hat * self.scaling_factor

    @property
    def beta(self):
        return self.beta_hat * (self.scaling_factor ** 2)

    @property
    def omega(self):
        return self.omega_hat * self.scaling_factor

    @property
    def r(self):
        return self.r_hat * (self.scaling_factor ** 2)

    @property
    def B(self):
        return self.B_hat * (self.scaling_factor ** 2)
    #============================================

    def forward(self, t):               # t shape (N, 1)
        if self.use_fourier:            # Fourier feature 사용 여부
            phi = self.enc(t)           # (N, 2m)
            x = torch.cat([t, phi], dim=-1)
        else:
            x = t
        return self.net(x)              # (N, 1)

**Loss Computation**

In [ ]:
#====================== Duffing Equation =================================
def loss_computation(tau, tau_colloc, model, u_obs, tau0, u0, u0_dot, tauT, uT, uT_dot,
                     lambda_data, lambda_pde, lambda_ic, lambda_bc):
  # Data loss
  u_pred_data = model(tau).reshape_as(u_obs)
  mse_data = nn.functional.mse_loss(u_pred_data, u_obs)

  # PDE loss
  tau_colloc = tau_colloc.clone().detach().requires_grad_(True)   
  u_pred_pde = model(tau_colloc)

  # First-order derivative
  du_dtau = torch.autograd.grad(
      u_pred_pde, tau_colloc,
      grad_outputs=torch.ones_like(u_pred_pde),
      create_graph=True, retain_graph=True
      )[0]

  # Second-order derivative
  d2u_dtau2 = torch.autograd.grad(
      du_dtau, tau_colloc,
      grad_outputs=torch.ones_like(du_dtau),
      create_graph=True)[0]

  # driven eqn parameters
  alpha_hat = model.alpha_hat  
  beta_hat = model.beta_hat  
  omega_hat = model.omega_hat 
  phi = model.phi
  B_hat = model.B_hat
  r_hat = model.r_hat

  # Residual
  residual = d2u_dtau2 + alpha_hat * du_dtau + beta_hat * u_pred_pde + r_hat * u_pred_pde**3 - B_hat * torch.cos(omega_hat * tau_colloc + phi)

  # PDE loss (MSE_PDE)
  mse_pde = nn.functional.mse_loss(residual, torch.zeros_like(residual))

  # IC loss
  u0_pred = model(tau0).reshape_as(u0)
  du0_dtau = torch.autograd.grad(
      u0_pred, tau0,
      grad_outputs=torch.ones_like(u0_pred),
      create_graph=True)[0].reshape_as(u0_dot)

  mse_ic = nn.functional.mse_loss(u0_pred, u0) + nn.functional.mse_loss(du0_dtau, u0_dot)

  # BC loss
  uT_pred = model(tauT).reshape_as(uT)
  duT_dtau = torch.autograd.grad(
    uT_pred, tauT,
    grad_outputs=torch.ones_like(uT_pred),
    create_graph=True)[0].reshape_as(uT_dot)

  mse_bc = nn.functional.mse_loss(uT_pred, uT) + nn.functional.mse_loss(duT_dtau, uT_dot)

  # total loss
  total_loss = lambda_data * mse_data + lambda_pde * mse_pde + lambda_ic * mse_ic + lambda_bc * mse_bc

  scaling_factor = model.scaling_factor

  alpha = alpha_hat * scaling_factor
  beta = beta_hat * scaling_factor**2
  omega = omega_hat * scaling_factor
  r = r_hat * scaling_factor**2
  B = B_hat * scaling_factor**2

  return total_loss, mse_data, mse_pde, mse_ic, mse_bc, lambda_data, lambda_pde, lambda_ic, lambda_bc, u_pred_data, alpha, beta, omega, r, B


**Configuration (cfg)**

In [ ]:
@dataclass
class TrainCfg:
  # Network architecture
  n_hidden: int = 4
  n_neurons: int = 128
  use_fourier: bool = False
  fourier_m: int = 3
  fourier_sigma: float = 1.0

  # Training parameters
  n_epochs: int = 5000
  lr: float = 1e-3
  n_colloc: int = 1000

  lambda_data: float = 1.0
  lambda_fixed: float = 0.05
  lambda_ic: float = 0.05
  lambda_bc: float = 0.05

  # Optimizer
  optimizer: str = 'Adam'
  weight_decay: float = 1e-2   # AdamW (1e-4 ~ 1e-2)

  # L-BFGS
  use_lbfgs: bool = False 
  lbfgs_epochs = 4  
  lbfgs_max_iter = 25 
  lbfgs_lr = 1.0
  lbfgs_history = 10

  # Runtime settings
  print_every: int = 500
  use_best_state_adam: bool = True
  device: Optional[torch.device] = None

# ---------- Adjust training settings here ----------
cfg = TrainCfg(
    n_hidden=4,
    n_neurons=128,
    use_fourier=False,
    fourier_m=3,
    fourier_sigma=1.0,
    n_epochs=10000,
    lr=1e-3,
    n_colloc=1024,
    lambda_data = 1.0,
    lambda_fixed = 1.0, # weight for PDE residual loss
    lambda_ic = 0.0,
    lambda_bc = 0.0,
    use_best_state_adam=True,
    print_every=100
  )

# ---------- Save experiment settings ----------
def _json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.device):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    return value


def _cfg_to_dict(cfg):
    keys = [
        "n_hidden", "n_neurons", "use_fourier", "fourier_m", "fourier_sigma",
        "n_epochs", "lr", "n_colloc",
        "lambda_data", "lambda_fixed", "lambda_ic", "lambda_bc",
        "optimizer", "weight_decay", "use_lbfgs", "lbfgs_epochs",
        "lbfgs_max_iter", "lbfgs_lr", "lbfgs_history", "print_every",
        "use_best_state_adam", "device",
    ]
    return {key: _json_safe(getattr(cfg, key)) for key in keys if hasattr(cfg, key)}


def _record_summary(records):
    rows = []
    for record in records:
        n_samples = int(len(record.x))
        duration_sec = float(n_samples / record.fs) if record.fs else None
        rows.append({
            "idx": int(record.idx),
            "group": str(record.group),
            "filename": str(record.filename),
            "fs": float(record.fs),
            "n_samples": n_samples,
            "duration_sec": duration_sec,
        })
    return rows


def _window_summary(file_windows):
    rows = []
    for t_win, x_win, fs, tag, filename in file_windows:
        rows.append({
            "tag": str(tag),
            "filename": str(filename),
            "fs": float(fs),
            "n_windows": int(t_win.shape[0]),
            "n_points_per_window": int(t_win.shape[1]) if t_win.ndim >= 2 else 0,
        })
    return rows


def build_experiment_config():
    import platform
    import sys
    from datetime import datetime

    device = cfg.device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    omega_init = 2.0 * np.pi
    beta_init = omega_init**2
    B_init = 0.5

    file_groups = {
        "Healthy": globals().get("file_H", []),
        "Wheeze": globals().get("file_W", []),
        "Crackle": globals().get("file_C", []),
    }
    window_groups = {
        "Healthy": globals().get("window_H", []),
        "Wheeze": globals().get("window_W", []),
        "Crackle": globals().get("window_C", []),
    }

    config = {
        "run": {
            "run_name": RUN_NAME,
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "root_dir": ROOT_DIR,
        },
        "paths": {
            "data_dir": DATA_DIR,
            "selected_files_csv": SELECTED_FILES_CSV,
            "results_dir": RESULTS_DIR,
            "pdf_dir": PDF_DIR,
            "saved_model_dir": SAVED_MODEL_DIR,
            "run_plot_dir": RUN_PLOT_DIR,
            "smoothing_plot_dir": SMOOTHING_PLOT_DIR,
            "ode_plot_dir": ODE_PLOT_DIR,
            "loss_history_dir": SAVED_MODEL_DIR / "loss_history",
            "loss_curve_dir": RUN_PLOT_DIR / "loss_vs_epoch_curve",
            "param_csv_path": PARAM_CSV_PATH,
            "model_params_csv_path": MODEL_PARAMS_CSV_PATH,
            "window_params_csv_path": WINDOW_PARAMS_CSV_PATH,
            "experiment_setting_txt_path": EXPERIMENT_SETTING_TXT_PATH,
            "experiment_config_json_path": EXPERIMENT_CONFIG_JSON_PATH,
            "eval_metric_txt_path": EVAL_METRIC_TXT_PATH,
            "file_list_txt_path": FILE_LIST_TXT_PATH,
        },
        "data": {
            "file_counts": {name: len(records) for name, records in file_groups.items()},
            "files": {name: _record_summary(records) for name, records in file_groups.items()},
        },
        "smoothing": {
            "method": SMOOTHING_METHOD,
            "win_size": SMOOTHING_WIN_SIZE,
            "polyorder": SMOOTHING_POLYORDER,
            "output_dir": SMOOTHING_PLOT_DIR,
        },
        "windowing": {
            "win_sec": WIN_SEC,
            "overlap": MY_OVERLAP,
            "target_pts": 441,
            "short_policy": "drop",
            "window_counts": {
                name: int(sum(row[0].shape[0] for row in windows))
                for name, windows in window_groups.items()
            },
            "windows": {name: _window_summary(windows) for name, windows in window_groups.items()},
        },
        "model": {
            "class_name": "MlpModel",
            "in_dim": 1,
            "scaling_factor": 1.0 / WIN_SEC,
            "fourier_features": {
                "enabled": bool(cfg.use_fourier),
                "fourier_m": int(cfg.fourier_m) if cfg.use_fourier else None,
                "fourier_sigma": float(cfg.fourier_sigma) if cfg.use_fourier else None,
            },
            "initial_parameters": {
                "omega_init": float(omega_init),
                "beta_init": float(beta_init),
                "B_init": float(B_init),
                "alpha_r_phi_source": "MlpModel defaults",
            },
        },
        "training": _cfg_to_dict(cfg),
        "randomness": {
            "seed": SEED,
        },
        "environment": {
            "python": sys.version.replace("\n", " "),
            "platform": platform.platform(),
            "torch": torch.__version__,
            "cuda_available": bool(torch.cuda.is_available()),
            "device": str(device),
        },
    }
    return _json_safe(config)


def save_experiment_settings(config=None, txt_path=None, json_path=None):
    import json

    if config is None:
        config = build_experiment_config()
    txt_path = Path(txt_path or EXPERIMENT_SETTING_TXT_PATH)
    json_path = Path(json_path or EXPERIMENT_CONFIG_JSON_PATH)
    txt_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.parent.mkdir(parents=True, exist_ok=True)

    json_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

    lines = []
    lines.append(config["run"]["run_name"])
    lines.append("")
    lines.append("Experiment Setting")
    lines.append("")
    lines.append(f"created_at: {config['run']['created_at']}")
    lines.append(f"root_dir: {config['run']['root_dir']}")
    lines.append(f"seed: {config['randomness']['seed']}")
    lines.append(f"device: {config['environment']['device']}")
    lines.append(f"torch: {config['environment']['torch']}")
    lines.append("")
    lines.append("[Paths]")
    for key, value in config["paths"].items():
        lines.append(f"{key}: {value}")
    lines.append("")
    lines.append("[Data]")
    for group, count in config["data"]["file_counts"].items():
        lines.append(f"{group}: {count} files")
        for row in config["data"]["files"][group]:
            lines.append(
                f"  - {row['filename']} | fs={row['fs']} | "
                f"samples={row['n_samples']} | duration={row['duration_sec']:.2f}s"
            )
    lines.append("")
    lines.append("[Smoothing]")
    for key, value in config["smoothing"].items():
        lines.append(f"{key}: {value}")
    lines.append("")
    lines.append("[Windowing]")
    lines.append(f"win_sec: {config['windowing']['win_sec']}")
    lines.append(f"overlap: {config['windowing']['overlap']}")
    lines.append(f"target_pts: {config['windowing']['target_pts']}")
    lines.append(f"short_policy: {config['windowing']['short_policy']}")
    for group, count in config["windowing"]["window_counts"].items():
        lines.append(f"{group}: {count} windows")
    lines.append("")
    lines.append("[Model]")
    for key, value in config["model"].items():
        lines.append(f"{key}: {value}")
    lines.append("")
    lines.append("[Training]")
    for key, value in config["training"].items():
        lines.append(f"{key}: {value}")

    txt_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"[SAVE] Experiment setting txt -> {txt_path}")
    print(f"[SAVE] Experiment config json -> {json_path}")
    return txt_path, json_path


EXPERIMENT_CONFIG = build_experiment_config()
EXPERIMENT_SETTING_TXT, EXPERIMENT_CONFIG_JSON = save_experiment_settings(EXPERIMENT_CONFIG)



**Training Function**

In [ ]:
def train_pinn(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    tau_train: torch.Tensor,
    u_train: torch.Tensor,
    *,
    cfg: TrainCfg,
    tau0,
    u0,
    u0_dot,
    tauT,
    uT,
    uT_dot):

    # Environment Setup
    device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    tau_train = tau_train.to(device)
    u_train = u_train.to(device)
    tau0 = tau0.to(device)
    tauT = tauT.to(device)
    u0 = u0.to(device)
    uT = uT.to(device)
    u0_dot = u0_dot.to(device).unsqueeze(-1)
    uT_dot = uT_dot.to(device).unsqueeze(-1)

    history = []

    final_ep_u_pred = None

    # Best Adam snapshot tracking
    best_total = float('inf')  
    best_state_adam = None  
    best_u_pred_np = None     
    best_log = None

    # train loop (low-discrepancy Sobol collocation sampling)
    for ep in range(1, cfg.n_epochs + 1):
        sobol_ = SobolEngine(dimension=1, scramble=True, seed=SEED + ep)  
        sobol = sobol_.draw(cfg.n_colloc).to(device)   
        tau_colloc = tau0 + (tauT - tau0) * sobol

        lambda_pde_current = cfg.lambda_fixed

        # loss computation
        total_loss, mse_u, mse_pde, mse_ic, mse_bc, lambda_data, lambda_pde, lambda_ic, lambda_bc, u_pred_data, alpha, beta, omega, r, B = loss_computation(
            tau_train, tau_colloc, model, u_train, tau0, u0, u0_dot, tauT, uT, uT_dot, cfg.lambda_data, lambda_pde_current, cfg.lambda_ic, cfg.lambda_bc)

        # backpropagation, parameter update
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        final_ep_u_pred = u_pred_data.detach().cpu().numpy().squeeze()
        total = total_loss.item() 

        # best snapshot tracking
        if total < best_total:
            best_total = total
            best_state_adam = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_u_pred_np = final_ep_u_pred

            with torch.no_grad():
                lam_data_loss = lambda_data * mse_u.item()
                lam_pde_loss = lambda_pde * mse_pde.item()
                lam_ic_loss  = lambda_ic * mse_ic.item()
                lam_bc_loss  = lambda_bc * mse_bc.item()
                best_log = (
                    f"[{ep:5d}/{cfg.n_epochs}]\n"
                    f"------------------------------\n"
                    f" total_loss_adam(w)={total_loss.item():.2e} |\n"
                    f"------------------------------\n"
                    f" λ_data={lambda_data:.2e} |"
                    f" mse_u={mse_u.item():.2e} |"
                    f" λ*mse_u={lam_data_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" λ_pde={lambda_pde:.2e} |"
                    f" mse_pde={mse_pde.item():.2e} |"
                    f" λ*mse_pde={lam_pde_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" λ_IC={lambda_ic:.2e} |"
                    f" mse_IC={mse_ic.item():.2e} |"
                    f" λ_IC*mse_IC={lam_ic_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" λ_BC={lambda_bc:.2e} |"
                    f" mse_BC={mse_bc.item():.2e} |"
                     f" λ_BC*mse_BC={lam_bc_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" alpha={alpha.item():.2e} |"
                    f" beta={beta.item():.2e} |"
                    f" omega={omega.item():.2e} |"
                    f" γ={r.item():.2e} |"
                    f" B={B.item():.2e} |\n"
                    )

        history.append({
            'epoch': ep,
            'total_loss': total_loss.item(),
            'mse_u': mse_u.item(),
            'mse_pde': mse_pde.item(),
            'mse_IC': mse_ic.item(),
            'mse_BC': mse_bc.item(),
            'weighted_mse_u': lambda_data * mse_u.item(),
            'weighted_mse_pde': lambda_pde * mse_pde.item(),
            'weighted_mse_IC': lambda_ic * mse_ic.item(),
            'weighted_mse_BC': lambda_bc * mse_bc.item(),
            'lambda_data': lambda_data,
            'lambda_pde': lambda_pde,
            'lambda_IC': lambda_ic,
            'lambda_BC': lambda_bc,
            'alpha': alpha.item(),
            'beta': beta.item(),
            'omega': omega.item(),
            'r': r.item(),
            'B': B.item()
        })

    if best_log is not None:
        print(best_log, flush=True)

    if cfg.use_best_state_adam and best_state_adam is not None:
        model.load_state_dict(best_state_adam)
        if best_u_pred_np is not None:
            final_ep_u_pred = best_u_pred_np

    else:
        with torch.no_grad():
            lam_data_loss = lambda_data * mse_u.item()
            lam_pde_loss = lambda_pde * mse_pde.item()
            lam_ic_loss  = lambda_ic * mse_ic.item()
            lam_bc_loss  = lambda_bc * mse_bc.item()
            print(f"[{cfg.n_epochs:5d}/{cfg.n_epochs}] [Final Epoch]\n"
                f"------------------------------\n"
                f" total_loss_adam(w)={total_loss.item():.2e} |\n"
                f"------------------------------\n"
                f" λ_data={lambda_data:.2e} |"
                f" mse_u={mse_u.item():.2e} |"
                f" λ*mse_u={lam_data_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" λ_pde={lambda_pde:.2e} |"
                f" mse_pde={mse_pde.item():.2e} |"
                f" λ*mse_pde={lam_pde_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" λ_IC={lambda_ic:.2e} |"
                f" mse_IC={mse_ic.item():.2e} |"
                f" λ_IC*mse_IC={lam_ic_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" λ_BC={lambda_bc:.2e} |"
                f" mse_BC={mse_bc.item():.2e} |"
                f" λ_BC*mse_BC={lam_bc_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" alpha={alpha.item():.2e} |"
                f" beta={beta.item():.2e} |"
                f" omega={omega.item():.2e} |"
                f" γ={r.item():.2e} |"
                f" B={B.item():.2e} |\n",
                flush=True
                )

    print(f"[Completed] best_total={best_total:.3e}\n", flush=True)

    return history, final_ep_u_pred

**Train One Window**

In [ ]:
def train_one_window(t_input, u_input, cfg) -> Tuple[nn.Module, List[Dict[str, float]], np.ndarray]:
  t_raw = t_input.cpu().numpy().squeeze()
  x_raw = u_input.cpu().numpy().squeeze()

  omega_init = 2.0 * np.pi
  beta_init = omega_init**2
  B_init = 0.5
  scaling_factor = 1 / WIN_SEC

  tau = scaling_factor * t_input 
  tau_raw = tau.cpu().numpy().squeeze()

  h = float(np.diff(tau_raw).mean()) 

  device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = MlpModel(
      in_dim = 1,
      n_hidden = cfg.n_hidden,
      n_neurons = cfg.n_neurons,
      beta_init = beta_init,
      omega_init = omega_init, 
      B_init = B_init, 
      fourier_m = cfg.fourier_m,      
      fourier_sigma = cfg.fourier_sigma,
      use_fourier = cfg.use_fourier,
      scaling_factor = scaling_factor
      )

  model.to(device)

  alpha_init = model.alpha.item()
  r_init = model.r.item()
  phi_init = model.phi.item()
  
  tau_train = tau.to(device)
  u_train = u_input.to(device)

  print(f"omega_init: {model.omega.item():.4f}")
  print(f"beta_init: {model.beta.item():.4f}")
  print(f"B_init: {model.B.item():.4f}\n")

  # Stage 1: Adam optimization
  if cfg.optimizer == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
  elif cfg.optimizer == "AdamW":
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
  else:
    raise ValueError(f"Unsupported optimizer: {cfg.optimizer}")

  tau0 = tau_train[:1].clone().detach().to(device).requires_grad_(True)  
  tauT = tau_train[-1:].clone().detach().to(device).requires_grad_(True) 
  u0 = u_train[:1].clone()    
  uT = u_train[-1:].clone()   

  # Target derivatives in physical units
  if len(x_raw) >= 3:
      du0 = (-3*x_raw[0] + 4*x_raw[1] - x_raw[2]) / (2*h)
      duT = ( 3*x_raw[-1] - 4*x_raw[-2] + x_raw[-3]) / (2*h)
  else:
      du0 = (x_raw[1]  - x_raw[0])  / h
      duT = (x_raw[-1] - x_raw[-2]) / h
  u0_dot = torch.tensor([[du0]], device=device, dtype=u_train.dtype)
  uT_dot = torch.tensor([[duT]], device=device, dtype=u_train.dtype)

  print(
      "[IC/BC]:",
      f"u0={u0.detach().cpu().squeeze().item():.2e} |",
      f"u0'={u0_dot.detach().cpu().squeeze().item():.2e} |",
      f"uT={uT.detach().cpu().squeeze().item():.2e} |",
      f"uT'={uT_dot.detach().cpu().squeeze().item():.2e}\n",
      sep=" "
  )

  # Stage 1: Adam optimization
  hist1, _ = train_pinn(
      model, optimizer,
      tau_train,
      u_train,
      cfg=cfg,
      tau0=tau0,
      u0=u0,
      u0_dot=u0_dot,
      tauT=tauT,
      uT=uT,
      uT_dot=uT_dot
  )

  adam_final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
  lbfgs_failed = False
  lbfgs_fail_reason = ""

  # Stage 2: L-BFGS fine-tuning
  hist2 = []

  if cfg.use_lbfgs:
    lbfgs = torch.optim.LBFGS(model.parameters(),
                                  lr=cfg.lbfgs_lr,
                                  history_size=cfg.lbfgs_history,
                                  max_iter=cfg.lbfgs_max_iter,
                                  line_search_fn="strong_wolfe",
                                  tolerance_grad=1e-7,  
                                  tolerance_change=1e-9, 
                                  )

    print("Starting L-BFGS ...")

    lambda_pde_lbfgs = cfg.lambda_fixed

    for step in range(1, cfg.lbfgs_epochs + 1):
      sobol_ = SobolEngine(dimension=1, scramble=True, seed=SEED + step)  
      sobol = sobol_.draw(cfg.n_colloc).to(device) 
      tau_colloc = tau0 + (tauT - tau0) * sobol

      def closure():
        lbfgs.zero_grad()
        total_loss, mse_u, mse_pde, mse_ic, mse_bc, lambda_data, lambda_pde, lambda_ic, lambda_bc, u_pred_data, alpha, beta, omega, r, B  = loss_computation(
            tau_train,
            tau_colloc,
            model,
            u_train,
            tau0,
            u0,
            u0_dot,
            tauT,
            uT,
            uT_dot,
            lambda_data=cfg.lambda_data,
            lambda_pde=lambda_pde_lbfgs,
            lambda_ic=cfg.lambda_ic,
            lambda_bc=cfg.lambda_bc,
        )
        total_loss.backward()

        closure.mse_u = mse_u.item()
        closure.mse_pde = mse_pde.item()
        closure.mse_ic = mse_ic.item()
        closure.mse_bc = mse_bc.item()
        closure.lambda_data = lambda_data
        closure.lambda_pde = lambda_pde
        closure.lambda_ic = lambda_ic
        closure.lambda_bc = lambda_bc
        closure.alpha = alpha.item()
        closure.beta = beta.item()
        closure.omega = omega.item()
        closure.r = r.item()
        closure.B = B.item()
        return total_loss

      loss_value = lbfgs.step(closure)
      loss_scalar = loss_value.item()
      lbfgs_is_finite = all([
          math.isfinite(loss_scalar),
          math.isfinite(closure.mse_u),
          math.isfinite(closure.mse_pde),
          math.isfinite(closure.mse_ic),
          math.isfinite(closure.mse_bc),
          math.isfinite(closure.alpha),
          math.isfinite(closure.beta),
          math.isfinite(closure.omega),
          math.isfinite(closure.r),
          math.isfinite(closure.B),
      ])

      if not lbfgs_is_finite:
        lbfgs_failed = True
        lbfgs_fail_reason = f"non-finite value at L-BFGS step {step}"
        model.load_state_dict(adam_final_state)
        hist2 = []
        print(f"[WARN] L-BFGS failed ({lbfgs_fail_reason}). Restored Adam final state.\n")
        break

      if step == cfg.lbfgs_epochs:
        lam_data_loss = closure.lambda_data * closure.mse_u
        lam_pde_loss = closure.lambda_pde * closure.mse_pde
        lam_ic_loss = closure.lambda_ic * closure.mse_ic
        lam_bc_loss = closure.lambda_bc * closure.mse_bc
        print(f"L-BFGS [{step}/{cfg.lbfgs_epochs}]\n"
              f"------------------------------\n"
              f" total_loss={loss_value.item():.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_data={closure.lambda_data:.2e} |"
              f" mse_u={closure.mse_u:.2e} |"
              f" λ*mse_u={lam_data_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_pde={closure.lambda_pde:.2e} |"
              f" mse_pde={closure.mse_pde:.2e} |"
              f" λ*mse_pde={lam_pde_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_IC={closure.lambda_ic:.2e} |"
              f" mse_IC={closure.mse_ic:.2e} |"
              f" λ_IC*mse_IC={lam_ic_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_BC={closure.lambda_bc:.2e} |"
              f" mse_BC={closure.mse_bc:.2e} |"
              f" λ_BC*mse_BC={lam_bc_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" alpha={closure.alpha:.2e} |"
              f" beta={closure.beta:.2e} |"
              f" omega={closure.omega:.2e} |"
              f" γ={closure.r:.2e} |"
              f" B={closure.B:.2e} |\n"
              )

      hist2.append({
          'epoch': len(hist1) + step, 
          'lambda_data': closure.lambda_data,
          'lambda_pde': closure.lambda_pde,
          'lambda_IC' : closure.lambda_ic,
          'lambda_BC' : closure.lambda_bc,
          'total_loss': loss_value.item(),
          'mse_u': closure.mse_u,
          'mse_pde': closure.mse_pde,
          'mse_IC' : closure.mse_ic,
          'mse_BC' : closure.mse_bc,
          'weighted_mse_u': closure.lambda_data * closure.mse_u,
          'weighted_mse_pde': closure.lambda_pde * closure.mse_pde,
          'weighted_mse_IC': closure.lambda_ic * closure.mse_ic,
          'weighted_mse_BC': closure.lambda_bc * closure.mse_bc,
          'alpha': closure.alpha,
          'beta': closure.beta,
          'omega': closure.omega,
          'r': closure.r,
          'B': closure.B
      })
    hist = hist1 if lbfgs_failed else (hist1 + hist2)
  else:
    hist = hist1

  with torch.no_grad():
    device_ = next(model.parameters()).device
    final_u_pred = model(tau.to(device_)).detach().cpu().numpy().squeeze()

  return model, hist, final_u_pred, omega_init, B_init, beta_init, alpha_init, r_init, phi_init

**Add ODE forward to the prediction plot**

In [ ]:
def duffing_forward(t, u0, v0, alpha, beta, r, B, omega, phi=0.0, method="RK45", rtol=1e-7, atol=1e-9):
    """
    u'' + alpha*u' + beta*u + r*u^3 = B*cos(omega*tau + phi)
    """

    # RHS
    def rhs(ti, y):
        u, v = y 
        du = v
        dv = (
            - alpha * v
            - beta  * u
            - r     * u**3
            + B * np.cos(omega * ti + phi)
        )
        return [du, dv]

    sol = solve_ivp(
        rhs,
        t_span=(float(t[0]), float(t[-1])),  
        y0=[float(u0), float(v0)],  
        t_eval=t,  
        method=method,  # Default: explicit Runge-Kutta method of order 5(4)
        rtol=rtol,  
        atol=atol,  
    )

    if not sol.success:
        raise RuntimeError(f"ODE solve failed: {sol.message}")

    return sol.y[0]   

def fd_initial_conditions(t_raw, x_raw):
    dt = float(np.diff(t_raw).mean())  
    u0 = float(x_raw[0])

    if len(x_raw) >= 3:
        v0 = (-3*x_raw[0] + 4*x_raw[1] - x_raw[2]) / (2*dt) 
    else:
        v0 = (x_raw[1] - x_raw[0]) / dt

    return u0, float(v0)

def _to_numpy_1d(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().squeeze()
    return np.asarray(x).squeeze()

def plot_ode_forward_v0_comparison(model, t_raw, x_raw, scaling_factor, window_idx=None, save_path=None, compare_pinn_autograd_v0=False):
    """
    Plot PINN prediction, ground truth, and ODE forward using GT u0 + GT finite-difference v0.
    Set compare_pinn_autograd_v0=True to additionally show the old mixed-IC comparison.
    """
    t_raw = _to_numpy_1d(t_raw).astype(np.float64).ravel()
    x_raw = _to_numpy_1d(x_raw).astype(np.float64).ravel()
    tau_raw = scaling_factor * t_raw

    was_training = model.training
    model.eval()

    try:
        device = next(model.parameters()).device
        dtype = next(model.parameters()).dtype

        # Latest hat-domain parameters from the current model.
        alpha = float(model.alpha_hat.item())
        beta  = float(model.beta_hat.item())
        omega = float(model.omega_hat.item())
        r     = float(model.r_hat.item())
        B     = float(model.B_hat.item())
        phi   = float(model.phi.item()) if hasattr(model, "phi") else 0.0

        u0, v0_gt_fd_dt = fd_initial_conditions(t_raw, x_raw)
        v0_gt_fd = float(v0_gt_fd_dt / scaling_factor)  

        tau_tensor = torch.as_tensor(tau_raw[:, None], device=device, dtype=dtype)
        with torch.no_grad():
            u_pinn = model(tau_tensor).detach().cpu().numpy().squeeze()

        v0_pinn_autograd = None
        if compare_pinn_autograd_v0:
            tau0 = torch.tensor([[float(tau_raw[0])]], device=device, dtype=dtype, requires_grad=True)
            u0_pred = model(tau0)
            v0_pinn_autograd = torch.autograd.grad(
                u0_pred,
                tau0,
                grad_outputs=torch.ones_like(u0_pred),
                create_graph=False,
                retain_graph=False,
            )[0].detach().cpu().item()

        print("[ODE forward IC check]")
        ic_msg = (
            f"window_idx={window_idx} | "
            f"u0={u0:.6e} | "
            f"v0_gt_fd={v0_gt_fd:.6e}"
        )
        if compare_pinn_autograd_v0:
            ic_msg += (
                f" | v0_pinn_autograd={v0_pinn_autograd:.6e} | "
                f"abs_diff={abs(v0_gt_fd - v0_pinn_autograd):.6e}"
            )
        print(ic_msg)

        ode_gt_fd = None
        ode_pinn_ad = None
        ode_gt_fd_msg = ""
        ode_pinn_ad_msg = ""

        try:
            ode_gt_fd = duffing_forward(
                t=tau_raw,
                u0=u0,
                v0=v0_gt_fd,
                alpha=alpha,
                beta=beta,
                r=r,
                B=B,
                omega=omega,
                phi=phi,
            )
        except Exception as e:
            ode_gt_fd_msg = str(e)
            print(f"[ODE FAIL] GT finite-diff v0 | window_idx={window_idx} | {e}")

        if compare_pinn_autograd_v0:
            try:
                ode_pinn_ad = duffing_forward(
                    t=tau_raw,
                    u0=u0,
                    v0=v0_pinn_autograd,
                    alpha=alpha,
                    beta=beta,
                    r=r,
                    B=B,
                    omega=omega,
                    phi=phi,
                )
            except Exception as e:
                ode_pinn_ad_msg = str(e)
                print(f"[ODE FAIL] PINN autograd v0 | window_idx={window_idx} | {e}")

        plt.figure(figsize=(8, 3))
        plt.plot(tau_raw, u_pinn, label="PINN pred", color="tab:blue")
        plt.plot(tau_raw, x_raw, "--", label="Ground Truth", color="tab:orange")

        if ode_gt_fd is not None:
            plt.plot(tau_raw, ode_gt_fd, color="tab:green", label="ODE forward (GT u0 + GT fd v0)")
        else:
            plt.text(0.5, 0.88, "ODE GT-fd v0 FAILED", transform=plt.gca().transAxes,
                     ha="center", color="tab:green", fontsize=10)

        if compare_pinn_autograd_v0:
            if ode_pinn_ad is not None:
                plt.plot(tau_raw, ode_pinn_ad, color="tab:red", label="ODE forward (GT u0 + PINN autograd v0)")
            else:
                plt.text(0.5, 0.78, "ODE PINN-autograd v0 FAILED", transform=plt.gca().transAxes,
                         ha="center", color="tab:red", fontsize=10)

        title_suffix = "v0 comparison" if compare_pinn_autograd_v0 else "ODE forward"
        title = f"window-{window_idx} prediction | {title_suffix}" if window_idx is not None else f"prediction | {title_suffix}"
        plt.xlabel("Tau", fontsize=14)
        plt.ylabel("Amplitude", fontsize=14)
        plt.title(title, fontsize=14)
        plt.legend(fontsize=10)
        plt.tick_params(axis="both", which="major", labelsize=14)
        ax = plt.gca()
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        plt.tight_layout()

        if save_path is not None:
            save_plot_as_pdf(save_path)
        plt.show()
        plt.close()

        return {
            "tau_raw": tau_raw,
            "u_pinn": u_pinn,
            "ode_gt_fd": ode_gt_fd,
            "ode_pinn_autograd": ode_pinn_ad,
            "u0": u0,
            "v0_gt_fd": v0_gt_fd,
            "v0_pinn_autograd": v0_pinn_autograd,
            "alpha_hat": alpha,
            "beta_hat": beta,
            "omega_hat": omega,
            "r_hat": r,
            "B_hat": B,
            "phi": phi,
            "ode_gt_fd_msg": ode_gt_fd_msg,
            "ode_pinn_autograd_msg": ode_pinn_ad_msg,
        }
    finally:
        if was_training:
            model.train()

**Train All Windows**

In [ ]:
def train_all_windows(t_windows, x_windows, categories, cfg, num_train_windows=None, plot_n_windows=3, win_sec=WIN_SEC, file_idx=None, pdf_filename=None, models_dir=None):
  
  original_win_ids = list(range(1, len(t_windows) + 1))
  train_orders = list(range(1, len(t_windows) + 1))
  random_select_orders = list(range(1, len(t_windows) + 1))

  if num_train_windows is not None:  
      all_idx = list(range(len(t_windows)))
      random.shuffle(all_idx)
      train_idx_random = all_idx[:num_train_windows]
      random_select_order_map = {i: order for order, i in enumerate(train_idx_random, start=1)}
      train_idx = sorted(train_idx_random)

      t_windows = [t_windows[i] for i in train_idx]
      x_windows = [x_windows[i] for i in train_idx]
      categories = [categories[i] for i in train_idx]
      original_win_ids = [i + 1 for i in train_idx]
      train_orders = list(range(1, len(train_idx) + 1))
      random_select_orders = [random_select_order_map[i] for i in train_idx]

  selected_window_rows = [
      {
          'file_idx': file_idx,
          'category': cat,
          'train_order': train_order,
          'window': original_win_id,
          'random_select_order': random_select_order,
      }
      for cat, train_order, original_win_id, random_select_order
      in zip(categories, train_orders, original_win_ids, random_select_orders)
  ]

  if models_dir is not None:
    os.makedirs(models_dir, exist_ok=True)
    history_dir = os.path.join(models_dir, "loss_history")
    os.makedirs(history_dir, exist_ok=True)
    selection_dir = os.path.join(models_dir, "selected_windows")
    os.makedirs(selection_dir, exist_ok=True)
  else:
    history_dir = None
    selection_dir = None

  histories = []

  alpha_list = []  
  beta_list = []   
  omega_list = []  
  r_list = []      
  B_list = []      
  phi_list = []

  alpha_hat_list = []  
  beta_hat_list = []   
  omega_hat_list = []  
  r_hat_list = []      
  B_hat_list = []      

  alpha_init_list = []
  beta_init_list = []
  omega_init_list = []
  r_init_list = []
  B_init_list = []
  phi_init_list = []

  cat_tags = []  
  win_ids = []  
  train_order_ids = []
  random_select_order_ids = []
  final_u_preds = []   
  t_list = []  
  x_list = [] 

  cat_counts = Counter(categories)  
  cat_seen = defaultdict(int)   
  cat_plotted = defaultdict(int)   

  if selection_dir is not None and selected_window_rows:
      category_for_filename = str(selected_window_rows[0]['category']).replace(os.sep, '_')
      selection_path = os.path.join(selection_dir, f"selected_windows_file_{file_idx}_{category_for_filename}.csv")
      pd.DataFrame(selected_window_rows).to_csv(selection_path, index=False)
      print(f"[SAVE] {selection_path}")

  def relu_no_utt_forward_train(t, u0, alpha, beta, r, B, omega, phi=0.0,
                                method="RK45", rtol=1e-7, atol=1e-9):
    """alpha*u_tau + beta*u + r*u^3 = B*cos(omega*tau + phi)."""
    t = np.asarray(t, dtype=np.float64).ravel()
    alpha = float(alpha)
    beta = float(beta)
    r = float(r)
    B = float(B)
    omega = float(omega)
    phi = float(phi)

    if np.isclose(alpha, 0.0):
      raise RuntimeError("alpha is too close to zero for ReLU no-u'' first-order forward")

    def rhs(ti, y):
      u = y[0]
      du = (B * np.cos(omega * ti + phi) - beta * u - r * u**3) / alpha
      return [du]

    sol = solve_ivp(
      rhs,
      t_span=(float(t[0]), float(t[-1])),
      y0=[float(u0)],
      t_eval=t,
      method=method,
      rtol=rtol,
      atol=atol,
    )

    if not sol.success:
      raise RuntimeError(f"ReLU no-u'' ODE solve failed: {sol.message}")

    return sol.y[0]

  for idx, (cat, t_input, x_input, original_win_id, train_order, random_select_order) in enumerate(zip(categories, t_windows, x_windows, original_win_ids, train_orders, random_select_orders)):
    file_num = f"[File {file_idx}]"
    window_id = original_win_id
    if cat_seen[cat] == 0:
      print(f"\n===== {file_num} {cat} training =====")

    cat_seen[cat] += 1
    print(f"{file_num}[{cat}] train_order {train_order}/{cat_counts[cat]} | original window {window_id}")

    model, hist, final_u_pred, omega_init, B_init, beta_init, alpha_init, r_init, phi_init = train_one_window(t_input, x_input, cfg)

    t_raw = t_input.cpu().numpy().squeeze()
    x_raw = x_input.cpu().numpy().squeeze()

    scaling_factor = 1.0 / win_sec
    tau_raw = scaling_factor * t_raw

    u0, v0 = fd_initial_conditions(t_raw, x_raw)
    v0 = v0 / scaling_factor   # du/dtau

    alpha = model.alpha_hat.item()
    beta  = model.beta_hat.item()
    omega = model.omega_hat.item()
    r     = model.r_hat.item()
    B     = model.B_hat.item()
    phi   = model.phi.item()

    try:
        u_sim_original = duffing_forward(
            t=tau_raw,
            u0=u0,
            v0=v0,
            alpha=alpha,
            beta=beta,
            r=r,
            B=B,
            omega=omega,
            phi=phi,
        )
        original_ode_success = True
    except Exception as e:
        print(f"[Original ODE FAIL] [{cat}] window {cat_seen[cat]} | {e}")
        u_sim_original = None
        original_ode_success = False

    try:
        u_sim_relu_no_utt = relu_no_utt_forward_train(
            t=tau_raw,
            u0=u0,
            alpha=alpha,
            beta=beta,
            r=r,
            B=B,
            omega=omega,
            phi=phi,
        )
        relu_no_utt_ode_success = True
    except Exception as e:
        print(f"[ReLU no-u'' ODE FAIL] [{cat}] window {cat_seen[cat]} | {e}")
        u_sim_relu_no_utt = None
        relu_no_utt_ode_success = False

    alpha_list.append(model.alpha.item())
    beta_list.append(model.beta.item())
    omega_list.append(model.omega.item())
    r_list.append(model.r.item())
    B_list.append(model.B.item())

    alpha_hat_list.append(model.alpha_hat.item())
    beta_hat_list.append(model.beta_hat.item())
    omega_hat_list.append(model.omega_hat.item())
    r_hat_list.append(model.r_hat.item())
    B_hat_list.append(model.B_hat.item())

    phi_list.append(model.phi.item())

    alpha_init_list.append(alpha_init)
    beta_init_list.append(beta_init)
    omega_init_list.append(omega_init)
    r_init_list.append(r_init)
    B_init_list.append(B_init)
    phi_init_list.append(phi_init)
    cat_tags.append(cat)
    win_ids.append(window_id)
    train_order_ids.append(train_order)
    random_select_order_ids.append(random_select_order)   

    # plot loss
    ep           = [h['epoch']           for h in hist]
    total_loss   = [h['total_loss']      for h in hist]
    data_w_loss  = [h['lambda_data'] * h['mse_u']   for h in hist]
    ode_w_loss   = [h['lambda_pde']  * h['mse_pde'] for h in hist]
    ic_loss      = [h['mse_IC']          for h in hist]

    if models_dir is not None:
        model_path = os.path.join(models_dir, f"model_{file_idx}_{cat}_window_{window_id}.pt")
    else:
        model_path = f"model_{file_idx}_{cat}_window_{window_id}.pt"
    torch.save(model.state_dict(), model_path)
    print(f"[SAVE] {model_path}")

    if history_dir is not None:
        history_path = os.path.join(history_dir, f"loss_history_file_{file_idx}_{cat}_window_{window_id}.csv")
        pd.DataFrame(hist).to_csv(history_path, index=False)
        print(f"[SAVE] {history_path}")

    del model
    torch.cuda.empty_cache()

    histories.append({
        'loss': [h['total_loss'] for h in hist],
        'mse_u': [h['mse_u'] for h in hist],
        'mse_pde': [h['mse_pde'] for h in hist]
    })
    del hist   
    torch.cuda.empty_cache()

    t_list.append(t_raw)
    x_list.append(x_raw)

    if isinstance(final_u_pred, torch.Tensor):
      final_u_pred = final_u_pred.detach().cpu().numpy().squeeze()

    final_u_preds.append(final_u_pred)

    # Plot prediction curves for up to plot_n_windows samples from each category
    if cat_plotted[cat] < plot_n_windows:

      def _plot_prediction_with_ode(ode_values, ode_label, ode_success, ode_fail_text, file_tag, title_suffix):
        plt.figure(figsize=(8, 3))
        plt.plot(tau_raw, final_u_pred, label="PINN pred")
        plt.plot(tau_raw, x_raw, '--', label="Ground Truth")

        if ode_values is not None:
            plt.plot(tau_raw, ode_values, color="green", label=ode_label)
        else:
            plt.text(
                0.5, 0.9, ode_fail_text,
                transform=plt.gca().transAxes,
                ha="center", color="red", fontsize=12
            )

        title_msg = f"{cat} window-{window_id} prediction | {title_suffix}"
        if not ode_success:
            title_msg += " | ODE FAIL"

        plt.xlabel("Tau", fontsize=14)
        plt.ylabel("Amplitude", fontsize=14)
        plt.title(title_msg, fontsize=14)
        plt.legend(fontsize=12)
        plt.tick_params(axis='both', which='major', labelsize=14)
        ax = plt.gca()
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        plt.tight_layout()

        if pdf_filename is not None:
                  amplitude_plot_dir = os.path.join(pdf_filename, "amplitude_vs_time_plot")
                  os.makedirs(amplitude_plot_dir, exist_ok=True)
                  fname = f"{file_tag}_file_{file_idx}_{cat}_window_{window_id}.pdf"
                  full_path = os.path.join(amplitude_plot_dir, fname)
                  save_plot_as_pdf(full_path)

        plt.show()
        plt.close()
      '''
      _plot_prediction_with_ode(
        ode_values=u_sim_original,
        ode_label="ODE forward (ReLU w/ u'')",
        ode_success=original_ode_success,
        ode_fail_text="ORIGINAL ODE FAILED",
        file_tag="original_ode",
        title_suffix="ODE (w/ u'')",
      )'''

      _plot_prediction_with_ode(
        ode_values=u_sim_relu_no_utt,
        ode_label="ODE forward (ReLU w/o u'')",
        ode_success=relu_no_utt_ode_success,
        ode_fail_text="ReLU no-u'' ODE FAILED",
        file_tag="relu_no_utt_ode",
        title_suffix="ODE (w/o u'')",
      )

      # plot weighted loss
      plt.figure(figsize=(9,4.5))
      plt.semilogy(ep, total_loss,  '--', linewidth=2.0, alpha=0.9, label="total loss", zorder=3)
      plt.semilogy(ep, data_w_loss, label="λ·data loss", zorder=2)
      plt.semilogy(ep, ode_w_loss,  label="λ·ODE loss", zorder=2)
      #plt.semilogy(ep, ic_loss,     label="IC loss", zorder=2)
      plt.xlabel("Epoch")
      plt.ylabel("Loss")
      plt.legend()
      plt.tight_layout()
      if pdf_filename is not None:
              loss_curve_dir = os.path.join(pdf_filename, "loss_vs_epoch_curve")
              os.makedirs(loss_curve_dir, exist_ok=True)
              fname = f"file_{file_idx}_{cat}_window_{window_id}.pdf"
              full_path = os.path.join(loss_curve_dir, f"loss_lambda_{fname}")
              save_plot_as_pdf(full_path)
      plt.show()
      plt.close()

      cat_plotted[cat] += 1  

  return histories, alpha_list, beta_list, omega_list, r_list, B_list, alpha_hat_list, beta_hat_list, omega_hat_list, r_hat_list, B_hat_list, phi_list, cat_tags, win_ids, train_order_ids, random_select_order_ids, final_u_preds, t_list, x_list, omega_init_list, B_init_list, beta_init_list, alpha_init_list, r_init_list, phi_init_list   #, models


**Helper Function**

In [ ]:
# Convert NumPy arrays to lists of torch tensors
def numpy_to_torch_list(t_arr, x_arr):
    t_list, x_list = [], []
    for t_win, x_win in zip(t_arr, x_arr):
        t_list.append(torch.from_numpy(t_win.astype('float32')).unsqueeze(-1))
        x_list.append(torch.from_numpy(x_win.astype('float32')).unsqueeze(-1))
    return t_list, x_list

**Train All Files**

In [ ]:
def train_all_files(file_windows_list, cfg, category_label, plot_n_windows=3):
    results = []
    for file_idx, file_data in enumerate(file_windows_list, start=1):

        t_windows, x_windows = file_data[:2]
        
        print(f"\n==== File {file_idx}/{len(file_windows_list)} ====")

        t_win_list, x_win_list = numpy_to_torch_list(t_windows, x_windows)

        categories = [category_label] * len(t_win_list)

        # Train all windows for the current file
        histories_, alpha_list_vals, beta_list_vals, omega_list_vals, r_list_vals, B_list_vals, alpha_hat_list_vals, beta_hat_list_vals, omega_hat_list_vals, r_hat_list_vals, B_hat_list_vals, phi_list_vals, cat_tags_, win_ids_, train_order_ids_, random_select_order_ids_, final_u_preds_, t_list, x_list, omega_init_vals, B_init_vals, beta_init_vals, alpha_init_vals, r_init_vals, phi_init_vals = train_all_windows(
            t_win_list,
            x_win_list,
            categories,
            cfg,
            num_train_windows=1, # Use all available windows
            plot_n_windows=plot_n_windows,
            file_idx=file_idx,
            pdf_filename=pdf_dir,
            models_dir=SAVED_MODEL_DIR
        )

        results.append({
            'file_idx': file_idx,
            'cat_tags': cat_tags_,
            'win_ids': win_ids_,
            'train_order_ids': train_order_ids_,
            'random_select_order_ids': random_select_order_ids_,
            'histories': histories_,

            # raw
            'alpha_list_vals': alpha_list_vals,
            'beta_list_vals': beta_list_vals,
            'omega_list_vals': omega_list_vals,
            'r_list_vals': r_list_vals,
            'B_list_vals': B_list_vals,

            # hat
            'alpha_hat_list_vals': alpha_hat_list_vals, 
            'beta_hat_list_vals': beta_hat_list_vals, 
            'omega_hat_list_vals': omega_hat_list_vals, 
            'r_hat_list_vals': r_hat_list_vals, 
            'B_hat_list_vals': B_hat_list_vals,

            'phi_list_vals': phi_list_vals,

            'omega_init_vals': omega_init_vals,
            'B_init_vals': B_init_vals,
            'beta_init_vals': beta_init_vals,
            'alpha_init_vals': alpha_init_vals,
            'r_init_vals': r_init_vals,
            'phi_init_vals': phi_init_vals,
            
            'final_u_preds': final_u_preds_,
            't_list': t_list,
            'x_list': x_list,
        })

    return results


def build_window_params_df_from_results(results, group_name_map):
    rows = []
    for grp_key, train_results in results.items():
        class_name = group_name_map.get(grp_key, grp_key)
        for file_res in train_results:
            n = len(file_res['alpha_list_vals'])
            for i in range(n):
                hist = file_res['histories'][i] if i < len(file_res['histories']) else None
                row = {
                    'group_key': grp_key,
                    'class': class_name,
                    'file_idx': int(file_res['file_idx']),
                    'window': int(file_res['win_ids'][i]),
                    'train_order': int(file_res['train_order_ids'][i]) if 'train_order_ids' in file_res else np.nan,
                    'random_select_order': int(file_res['random_select_order_ids'][i]) if 'random_select_order_ids' in file_res else np.nan,
                    'alpha': float(file_res['alpha_list_vals'][i]),
                    'beta': float(file_res['beta_list_vals'][i]),
                    'omega': float(file_res['omega_list_vals'][i]),
                    'r': float(file_res['r_list_vals'][i]),
                    'B': float(file_res['B_list_vals'][i]),
                    'phi': float(file_res['phi_list_vals'][i]),
                    'alpha_hat': float(file_res['alpha_hat_list_vals'][i]),
                    'beta_hat': float(file_res['beta_hat_list_vals'][i]),
                    'omega_hat': float(file_res['omega_hat_list_vals'][i]),
                    'r_hat': float(file_res['r_hat_list_vals'][i]),
                    'B_hat': float(file_res['B_hat_list_vals'][i]),
                    'alpha_init': float(file_res['alpha_init_vals'][i]),
                    'beta_init': float(file_res['beta_init_vals'][i]),
                    'omega_init': float(file_res['omega_init_vals'][i]),
                    'r_init': float(file_res['r_init_vals'][i]),
                    'B_init': float(file_res['B_init_vals'][i]),
                    'phi_init': float(file_res['phi_init_vals'][i]),
                }
                if hist is not None and len(hist.get('loss', [])) > 0:
                    row['final_loss'] = float(hist['loss'][-1])
                    row['final_mse_u'] = float(hist['mse_u'][-1])
                    row['final_mse_pde'] = float(hist['mse_pde'][-1])
                else:
                    row['final_loss'] = np.nan
                    row['final_mse_u'] = np.nan
                    row['final_mse_pde'] = np.nan
                rows.append(row)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['class', 'file_idx', 'window']).reset_index(drop=True)
        df['window_global'] = df.groupby('class').cumcount() + 1
    return df


def save_window_params_csv(results, group_name_map, csv_path=WINDOW_PARAMS_CSV_PATH):
    df = build_window_params_df_from_results(results, group_name_map)
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f"[SAVE] window-level parameter CSV -> {csv_path} (n_rows={len(df)})")
    return df


**Run Train**

In [ ]:
file_windows_dict = {
    'H': window_H,
    'W': window_W,
    'C': window_C,
}

group_name_map = {
    'H': 'Healthy',
    'W': 'Wheeze',
    'C': 'Crackle',
}

results = {}  

for group_name, file_windows in file_windows_dict.items():  
    label = group_name_map[group_name]
    results[group_name] = {}
    print(f"\n===== Training {group_name_map[group_name]} =====")

    train_results = train_all_files(file_windows, cfg, category_label=label, plot_n_windows=10) 

    results[group_name] = train_results

window_params_df = save_window_params_csv(results, group_name_map, WINDOW_PARAMS_CSV_PATH)


**2D Scatter Plot: Alpha, Beta**

In [ ]:
def plot_alpha_beta(alpha_list_vals, beta_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):

    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(alpha_list_vals, beta_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")

    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\beta$")

    ax.set_title(title, fontsize=18)

    ax.grid(True)
    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()


**2D Scatter Plot: Beta, B**

In [ ]:
def plot_beta_B(beta_list_vals, B_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(beta_list_vals, B_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")

    ax.set_xlabel(r"$\beta$")
    ax.set_ylabel(r"$B$")

    ax.set_title(title, fontsize=18)

    ax.grid(True)
    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()


**2D scatter Plot: Alpha, Gamma**

In [ ]:
def plot_alpha_gamma(alpha_list_vals, r_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):

    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(alpha_list_vals, r_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")
    
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\gamma$")

    ax.set_title(title, fontsize=18)

    ax.grid(True)
    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**2D Scatter Plot: B, Omega**

In [ ]:
def plot_B_omega(B_list_vals, omega_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):

    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(B_list_vals, omega_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")

    ax.set_xlabel(r"$B$")
    ax.set_ylabel(r"$\omega$")
    ax.set_title(title, fontsize=18)
    ax.grid(True)
    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**2D Scatter Plot: B, Gamma**

In [ ]:
def plot_B_gamma(B_list_vals, r_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):

    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(B_list_vals, r_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")

    ax.set_xlabel(r"$B$")
    ax.set_ylabel(r"$\gamma$")
    ax.set_title(title, fontsize=18)
    ax.grid(True)
    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**3D Scatter Plot: Alpha, Beta, Omega**

In [ ]:
def plot_alpha_beta_omega(alpha_list_vals, beta_list_vals, omega_list_vals,
                                  cat_tags_, win_ids_, colors=None,
                                  title="method 2 (Normalization)", pdf_filename=None):

  if colors is None:
    colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

  fig = plt.figure(figsize=(6, 6))
  ax  = fig.add_subplot(111, projection='3d')

  for a, b, o, cat, _ in zip(alpha_list_vals, beta_list_vals, omega_list_vals, cat_tags_, win_ids_):
    ax.scatter(a, b, o, color=colors[cat], alpha=0.6)

  for lab, clr in colors.items():
      ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

  ax.set_xlabel("\u03B1")
  ax.set_ylabel("\u03B2")
  ax.set_zlabel("\u03C9")

  ax.set_title(title, fontsize=18)

  plt.subplots_adjust(left=0.05, right=0.55)
  ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

  plt.tight_layout()

  if pdf_filename:
      save_plot_as_pdf(pdf_filename)

  plt.show()

**3D Scatter Plot: Alpha, Beta, Gamma**

In [ ]:
def plot_alpha_beta_gamma(alpha_list_vals, beta_list_vals, r_list_vals,
                                  cat_tags_, win_ids_, colors=None,
                                  title="method 2 (Normalization)", pdf_filename=None):

  if colors is None:
    colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

  fig = plt.figure(figsize=(6, 6))
  ax  = fig.add_subplot(111, projection='3d')

  for a, b, o, cat, _ in zip(alpha_list_vals, beta_list_vals, r_list_vals, cat_tags_, win_ids_):
    ax.scatter(a, b, o, color=colors[cat], alpha=0.6)

  for lab, clr in colors.items():
      ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

  ax.set_xlabel("\u03B1")
  ax.set_ylabel("\u03B2")
  ax.set_zlabel("\u03B3")

  ax.set_title(title, fontsize=18)

  plt.subplots_adjust(left=0.05, right=0.55)
  ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

  plt.tight_layout()

  if pdf_filename:
      save_plot_as_pdf(pdf_filename)

  plt.show()

**3D Plot: Alpha, B, Gamma**

In [ ]:
def plot_alpha_B_gamma(alpha_list_vals, B_list_vals, r_list_vals,
                                  cat_tags_, win_ids_, colors=None,
                                  title="method 2 (Normalization)", pdf_filename=None):

  if colors is None:
    colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

  fig = plt.figure(figsize=(6, 6))
  ax  = fig.add_subplot(111, projection='3d')

  for a, b, o, cat, _ in zip(alpha_list_vals, B_list_vals, r_list_vals, cat_tags_, win_ids_):
    ax.scatter(a, b, o, color=colors[cat], alpha=0.6)

  for lab, clr in colors.items():
      ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

  ax.set_xlabel("\u03B1")
  ax.set_ylabel("B")
  ax.set_zlabel("\u03B3")

  ax.set_title(title, fontsize=18)

  plt.subplots_adjust(left=0.05, right=0.55)
  ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

  plt.tight_layout()

  if pdf_filename:
      save_plot_as_pdf(pdf_filename)

  plt.show()

**3D Plot: Beta, Omega, Gamma**

In [ ]:
def plot_beta_omega_gamma(beta_list_vals, omega_list_vals, r_list_vals,
                         cat_tags_, win_ids_, colors=None,
                         title="Beta-Omega-Gamma 3D Scatter", pdf_filename=None):

    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    fig = plt.figure(figsize=(6, 6))
    ax  = fig.add_subplot(111, projection='3d')

    for b, o, r_, cat, _ in zip(beta_list_vals, omega_list_vals, r_list_vals, cat_tags_, win_ids_):
        ax.scatter(b, o, r_, color=colors[cat], alpha=0.6)

    for lab, clr in colors.items():
        ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

    ax.set_xlabel("\u03B2")
    ax.set_ylabel("\u03C9")
    ax.set_zlabel("\u03B3")

    ax.set_title(title, fontsize=18)

    plt.subplots_adjust(left=0.05, right=0.55)
    ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**Plot Call**

In [ ]:
# Scatter plots from saved parameter CSV (no retraining required)

import os
from pathlib import Path

import pandas as pd

def _load_params_df_for_plots():
    csv_candidates = [Path(WINDOW_PARAMS_CSV_PATH), Path(PARAM_CSV_PATH)]
    params_csv_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])
    if not params_csv_path.exists():
        raise FileNotFoundError(
            "Saved parameter CSV was not found. Expected one of: "
            + ", ".join(str(path) for path in csv_candidates)
        )

    params_df = pd.read_csv(params_csv_path)
    if "category" not in params_df.columns and "class" in params_df.columns:
        params_df = params_df.rename(columns={"class": "category"})

    required_cols = {"category", "file_idx", "window", "alpha", "beta", "omega", "r", "B"}
    missing_cols = required_cols - set(params_df.columns)
    if missing_cols:
        raise ValueError(f"CSV is missing required columns: {sorted(missing_cols)}")

    params_df = params_df.copy()
    params_df["category"] = params_df["category"].astype(str).replace(
        {
            "Health": "Healthy",
            "health": "Healthy",
            "wheeze": "Wheeze",
            "crackle": "Crackle",
        }
    )
    params_df["window"] = pd.to_numeric(params_df["window"], errors="coerce")
    params_df["file_idx"] = pd.to_numeric(params_df["file_idx"], errors="coerce")
    for col in ["alpha", "beta", "omega", "r", "B"]:
        params_df[col] = pd.to_numeric(params_df[col], errors="coerce")

    params_df = params_df.dropna(subset=["category", "file_idx", "window", "alpha", "beta", "omega", "r", "B"])
    params_df = params_df.sort_values(["file_idx", "category", "window"]).reset_index(drop=True)

    known_categories = {"Healthy", "Wheeze", "Crackle"}
    unknown_categories = sorted(set(params_df["category"]) - known_categories)
    if unknown_categories:
        raise ValueError(f"CSV contains categories without scatter colors: {unknown_categories}")

    return params_df, params_csv_path


def _format_file_label(file_id):
    try:
        as_float = float(file_id)
        if as_float.is_integer():
            return str(int(as_float))
    except (TypeError, ValueError):
        pass
    return str(file_id)


def _file_sort_key(file_id):
    label = _format_file_label(file_id)
    try:
        return (0, int(label))
    except ValueError:
        return (1, label)


def _complete_file_ids_for_all(params_df):
    expected_categories = [
        cat for cat in ["Healthy", "Wheeze", "Crackle"]
        if cat in set(params_df["category"])
    ]
    window_counts = (
        params_df
        .groupby(["file_idx", "category"])["window"]
        .nunique()
        .unstack(fill_value=0)
    )
    expected_window_counts = window_counts.reindex(columns=expected_categories, fill_value=0)
    return expected_window_counts.index[
        (expected_window_counts > 0).all(axis=1)
        & (expected_window_counts.nunique(axis=1) == 1)
    ].tolist()


params_df, params_csv_path_for_plots = _load_params_df_for_plots()
base_pdf_dir = str(pdf_dir)
while os.path.basename(os.path.normpath(base_pdf_dir)) == "scatter_and_histogram":
    base_pdf_dir = os.path.dirname(os.path.normpath(base_pdf_dir))
scatter_histogram_dir = os.path.join(base_pdf_dir, "scatter_and_histogram")
os.makedirs(scatter_histogram_dir, exist_ok=True)

complete_file_ids_for_all = _complete_file_ids_for_all(params_df)
params_all_df = params_df[params_df["file_idx"].isin(complete_file_ids_for_all)].copy()
if params_all_df.empty:
    raise ValueError("No complete matched file sets are available for all plots.")

print(f"[LOAD] {params_csv_path_for_plots}")
print(f"[ROWS] {len(params_df)}")
print(f"[SAVE DIR] {scatter_histogram_dir}")
print(f"[CATEGORIES] {params_df['category'].value_counts().to_dict()}")
print(f"[ALL FILES] {[_format_file_label(file_id) for file_id in complete_file_ids_for_all]}")
print(f"[ALL ROWS] {len(params_all_df)}")

cat_tags_all = params_all_df["category"].tolist()
win_ids_all = params_all_df["window"].tolist()
alpha_all = params_all_df["alpha"].tolist()
beta_all = params_all_df["beta"].tolist()
omega_all = params_all_df["omega"].tolist()
r_all = params_all_df["r"].tolist()
B_all = params_all_df["B"].tolist()

plot_alpha_beta(alpha_all, beta_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\alpha$–$\beta$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_alpha_beta_all.pdf"))
plot_beta_B(beta_all, B_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\beta$–$B$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_beta_B_all.pdf"))
plot_alpha_gamma(alpha_all, r_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\alpha$–$\gamma$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_alpha_gamma_all.pdf"))
plot_B_omega(B_all, omega_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $B$–$\omega$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_B_omega_all.pdf"))
plot_B_gamma(B_all, r_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $B$–$\gamma$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_B_gamma_all.pdf"))
plot_alpha_beta_omega(alpha_all, beta_all, omega_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\alpha$–$\beta$–$\omega$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_alpha_beta_omega_all.pdf"))
plot_alpha_beta_gamma(alpha_all, beta_all, r_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\alpha$–$\beta$–$\gamma$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_alpha_beta_gamma_all.pdf"))
plot_alpha_B_gamma(alpha_all, B_all, r_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\alpha$–$B$–$\gamma$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_alpha_B_gamma_all.pdf"))
plot_beta_omega_gamma(beta_all, omega_all, r_all, cat_tags_all, win_ids_all, title=r"Duffing Equation $\beta$–$\omega$–$\gamma$ scatter", pdf_filename=os.path.join(scatter_histogram_dir, "scatter_beta_omega_gamma_all.pdf"))

unique_file_ids = sorted(params_df["file_idx"].unique().tolist(), key=_file_sort_key)
for file_id in unique_file_ids:
    df_file = params_df[params_df["file_idx"] == file_id]
    if df_file.empty:
        continue

    alpha_batch = df_file["alpha"].tolist()
    beta_batch = df_file["beta"].tolist()
    omega_batch = df_file["omega"].tolist()
    r_batch = df_file["r"].tolist()
    B_batch = df_file["B"].tolist()
    cat_tags_batch = df_file["category"].tolist()
    win_ids_batch = df_file["window"].tolist()

    file_label = _format_file_label(file_id)
    title_suffix = f"(File {file_label})"
    pdf_suffix = f"(File {file_label})"

    plot_alpha_beta(alpha_batch, beta_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\alpha$–$\beta$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_alpha_beta_file_{pdf_suffix}.pdf"))
    plot_beta_B(beta_batch, B_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\beta$–$B$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_beta_B_file_{pdf_suffix}.pdf"))
    plot_alpha_gamma(alpha_batch, r_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\alpha$–$\gamma$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_alpha_gamma_file_{pdf_suffix}.pdf"))
    plot_B_omega(B_batch, omega_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $B$–$\omega$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_B_omega_file_{pdf_suffix}.pdf"))
    plot_B_gamma(B_batch, r_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $B$–$\gamma$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_B_gamma_file_{pdf_suffix}.pdf"))
    plot_alpha_beta_omega(alpha_batch, beta_batch, omega_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\alpha$–$\beta$–$\omega$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_alpha_beta_omega_file_{pdf_suffix}.pdf"))
    plot_alpha_beta_gamma(alpha_batch, beta_batch, r_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\alpha$–$\beta$–$\gamma$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_alpha_beta_gamma_file_{pdf_suffix}.pdf"))
    plot_alpha_B_gamma(alpha_batch, B_batch, r_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\alpha$–$B$–$\gamma$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_alpha_B_gamma_file_{pdf_suffix}.pdf"))
    plot_beta_omega_gamma(beta_batch, omega_batch, r_batch, cat_tags_batch, win_ids_batch, title=fr"Duffing Equation $\beta$–$\omega$–$\gamma$ scatter {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"scatter_beta_omega_gamma_file_{pdf_suffix}.pdf"))

print(f"[DONE] Saved scatter PDFs from saved CSV -> {scatter_histogram_dir}")


**Plot Histogram**

In [ ]:
def plot_coefficient_histogram(values, cat_tags, coeff_name,
                               colors=None, bins=30, init_values=None, pdf_filename=None
                               ):
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 4))

    values_np = np.array(values)
    bin_edges = np.histogram_bin_edges(values_np, bins=bins)

    if coeff_name.startswith(r"$\beta$"):
        x_min = values_np.min()
        x_max = 40
        bin_edges = np.linspace(x_min, x_max, bins + 1) 

    elif coeff_name.startswith(r"$\gamma$"):
        x_min, x_max = -70, 5
        bin_edges = np.linspace(x_min, x_max, bins + 1)
    else:
        bin_edges = np.histogram_bin_edges(values_np, bins=bins)

    # Plot class-specific histograms
    for cls, clr in colors.items():
        vals = [v for v, tag in zip(values, cat_tags) if tag == cls]
        weights = np.ones_like(vals) / len(vals)
        plt.hist(vals, bins=bin_edges,
                    weights=weights,
                    alpha=0.6, label=cls, color=clr)
        
    # Initial value vertical dashed line
    if init_values is not None and len(init_values) > 0:
        init_val = init_values[0]  
        plt.axvline(init_val, color='gray', linestyle='--',
                    linewidth=2.0, alpha=0.8, label="Init value")

    plt.xlabel(coeff_name + " Values", fontsize=24)
    plt.ylabel("Probability", fontsize=24)
    plt.legend(fontsize=16, loc="upper center")
    plt.xticks(fontsize=22)
    plt.yticks(np.arange(0, 1.01, 0.1), fontsize=22)
    plt.grid(True, linestyle='--', alpha=0.3)
    if coeff_name.startswith(r"$\gamma$"):
        plt.ylim(0, 0.6)
    else:
        plt.ylim(0, 0.3)

    plt.tight_layout()

    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

# Histogram plots from saved parameter CSV (no retraining required)

import os
from pathlib import Path

import numpy as np
import pandas as pd

if "_load_params_df_for_plots" not in globals():
    def _load_params_df_for_plots():
        csv_candidates = [Path(WINDOW_PARAMS_CSV_PATH), Path(PARAM_CSV_PATH)]
        params_csv_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])
        if not params_csv_path.exists():
            raise FileNotFoundError(
                "Saved parameter CSV was not found. Expected one of: "
                + ", ".join(str(path) for path in csv_candidates)
            )

        params_df = pd.read_csv(params_csv_path)
        if "category" not in params_df.columns and "class" in params_df.columns:
            params_df = params_df.rename(columns={"class": "category"})

        required_cols = {"category", "file_idx", "window", "alpha", "beta", "omega", "r", "B"}
        missing_cols = required_cols - set(params_df.columns)
        if missing_cols:
            raise ValueError(f"CSV is missing required columns: {sorted(missing_cols)}")

        params_df = params_df.copy()
        params_df["category"] = params_df["category"].astype(str).replace(
            {
                "Health": "Healthy",
                "health": "Healthy",
                "wheeze": "Wheeze",
                "crackle": "Crackle",
            }
        )
        params_df["window"] = pd.to_numeric(params_df["window"], errors="coerce")
        params_df["file_idx"] = pd.to_numeric(params_df["file_idx"], errors="coerce")
        for col in ["alpha", "beta", "omega", "r", "B"]:
            params_df[col] = pd.to_numeric(params_df[col], errors="coerce")

        params_df = params_df.dropna(subset=["category", "file_idx", "window", "alpha", "beta", "omega", "r", "B"])
        params_df = params_df.sort_values(["file_idx", "category", "window"]).reset_index(drop=True)

        known_categories = {"Healthy", "Wheeze", "Crackle"}
        unknown_categories = sorted(set(params_df["category"]) - known_categories)
        if unknown_categories:
            raise ValueError(f"CSV contains categories without plot colors: {unknown_categories}")

        return params_df, params_csv_path

if "_format_file_label" not in globals():
    def _format_file_label(file_id):
        try:
            as_float = float(file_id)
            if as_float.is_integer():
                return str(int(as_float))
        except (TypeError, ValueError):
            pass
        return str(file_id)

if "_file_sort_key" not in globals():
    def _file_sort_key(file_id):
        label = _format_file_label(file_id)
        try:
            return (0, int(label))
        except ValueError:
            return (1, label)

if "_complete_file_ids_for_all" not in globals():
    def _complete_file_ids_for_all(params_df):
        expected_categories = [
            cat for cat in ["Healthy", "Wheeze", "Crackle"]
            if cat in set(params_df["category"])
        ]
        window_counts = (
            params_df
            .groupby(["file_idx", "category"])["window"]
            .nunique()
            .unstack(fill_value=0)
        )
        expected_window_counts = window_counts.reindex(columns=expected_categories, fill_value=0)
        return expected_window_counts.index[
            (expected_window_counts > 0).all(axis=1)
            & (expected_window_counts.nunique(axis=1) == 1)
        ].tolist()

params_df, params_csv_path_for_histograms = _load_params_df_for_plots()
base_pdf_dir = str(pdf_dir)
while os.path.basename(os.path.normpath(base_pdf_dir)) == "scatter_and_histogram":
    base_pdf_dir = os.path.dirname(os.path.normpath(base_pdf_dir))
scatter_histogram_dir = os.path.join(base_pdf_dir, "scatter_and_histogram")
os.makedirs(scatter_histogram_dir, exist_ok=True)

INIT_ALPHA = 0.5
INIT_OMEGA = 2.0 * np.pi
INIT_BETA = INIT_OMEGA**2
INIT_R = 1.0
INIT_B = 0.5


def _constant_init(n, value):
    return np.full(int(n), float(value), dtype=float).tolist()


def make_binary_cat_tags(cat_tags):
    return [
        "Pneumonia" if tag in ("Wheeze", "Crackle") else ("Healthy" if tag == "Health" else tag)
        for tag in cat_tags
    ]


binary_histogram_colors = {'Healthy': 'blue', 'Pneumonia': 'orange'}

complete_file_ids_for_all = _complete_file_ids_for_all(params_df)
params_all_df = params_df[params_df["file_idx"].isin(complete_file_ids_for_all)].copy()
if params_all_df.empty:
    raise ValueError("No complete matched file sets are available for all plots.")

print(f"[LOAD] {params_csv_path_for_histograms}")
print(f"[ROWS] {len(params_df)}")
print(f"[SAVE DIR] {scatter_histogram_dir}")

cat_tags_all = params_all_df["category"].tolist()
alpha_all = params_all_df["alpha"].tolist()
beta_all = params_all_df["beta"].tolist()
omega_all = params_all_df["omega"].tolist()
r_all = params_all_df["r"].tolist()
B_all = params_all_df["B"].tolist()

n_all = len(params_all_df)
alpha_init_all = _constant_init(n_all, INIT_ALPHA)
beta_init_all = _constant_init(n_all, INIT_BETA)
omega_init_all = _constant_init(n_all, INIT_OMEGA)
r_init_all = _constant_init(n_all, INIT_R)
B_init_all = _constant_init(n_all, INIT_B)

plot_coefficient_histogram(alpha_all, cat_tags_all, r"$\alpha$", init_values=alpha_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_alpha_all.pdf"))
plot_coefficient_histogram(alpha_init_all, cat_tags_all, r"$\alpha_{\mathrm{init}}$", pdf_filename=os.path.join(scatter_histogram_dir, "histogram_alpha_init_all.pdf"))
plot_coefficient_histogram(beta_all,  cat_tags_all, r"$\beta$", init_values=beta_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_beta_all.pdf"))
plot_coefficient_histogram(beta_init_all, cat_tags_all, r"$\beta_{\mathrm{init}}$", pdf_filename=os.path.join(scatter_histogram_dir, "histogram_beta_init_all.pdf"))
plot_coefficient_histogram(omega_all, cat_tags_all, r"$\omega$", init_values=omega_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_omega_all.pdf"))
plot_coefficient_histogram(omega_init_all, cat_tags_all, r"$\omega_{\mathrm{init}}$", pdf_filename=os.path.join(scatter_histogram_dir, "histogram_omega_init_all.pdf"))
plot_coefficient_histogram(r_all, cat_tags_all, r"$\gamma$", bins=30, init_values=r_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_r_all.pdf"))
plot_coefficient_histogram(r_init_all, cat_tags_all, r"$\gamma_{\mathrm{init}}$", pdf_filename=os.path.join(scatter_histogram_dir, "histogram_r_init_all.pdf"))
plot_coefficient_histogram(B_all, cat_tags_all, r"$B$", init_values=B_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_B_all.pdf"))
plot_coefficient_histogram(B_init_all, cat_tags_all, r"$B_{\mathrm{init}}$", pdf_filename=os.path.join(scatter_histogram_dir, "histogram_B_init_all.pdf"))

cat_tags_binary_all = make_binary_cat_tags(cat_tags_all)
plot_coefficient_histogram(alpha_all, cat_tags_binary_all, r"$\alpha$", colors=binary_histogram_colors, init_values=alpha_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_alpha_binary_all.pdf"))
plot_coefficient_histogram(alpha_init_all, cat_tags_binary_all, r"$\alpha_{\mathrm{init}}$", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_alpha_init_binary_all.pdf"))
plot_coefficient_histogram(beta_all, cat_tags_binary_all, r"$\beta$", colors=binary_histogram_colors, init_values=beta_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_beta_binary_all.pdf"))
plot_coefficient_histogram(beta_init_all, cat_tags_binary_all, r"$\beta_{\mathrm{init}}$", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_beta_init_binary_all.pdf"))
plot_coefficient_histogram(omega_all, cat_tags_binary_all, r"$\omega$", colors=binary_histogram_colors, init_values=omega_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_omega_binary_all.pdf"))
plot_coefficient_histogram(omega_init_all, cat_tags_binary_all, r"$\omega_{\mathrm{init}}$", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_omega_init_binary_all.pdf"))
plot_coefficient_histogram(r_all, cat_tags_binary_all, r"$\gamma$", colors=binary_histogram_colors, bins=30, init_values=r_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_r_binary_all.pdf"))
plot_coefficient_histogram(r_init_all, cat_tags_binary_all, r"$\gamma_{\mathrm{init}}$", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_r_init_binary_all.pdf"))
plot_coefficient_histogram(B_all, cat_tags_binary_all, r"$B$", colors=binary_histogram_colors, init_values=B_init_all, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_B_binary_all.pdf"))
plot_coefficient_histogram(B_init_all, cat_tags_binary_all, r"$B_{\mathrm{init}}$", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, "histogram_B_init_binary_all.pdf"))

unique_file_ids = sorted(params_df["file_idx"].unique().tolist(), key=_file_sort_key)
for file_id in unique_file_ids:
    df_file = params_df[params_df["file_idx"] == file_id]
    if df_file.empty:
        continue

    cat_tags_batch = df_file["category"].tolist()
    alpha_batch = df_file["alpha"].tolist()
    beta_batch = df_file["beta"].tolist()
    omega_batch = df_file["omega"].tolist()
    r_batch = df_file["r"].tolist()
    B_batch = df_file["B"].tolist()

    n_file = len(df_file)
    alpha_init_batch = _constant_init(n_file, INIT_ALPHA)
    beta_init_batch = _constant_init(n_file, INIT_BETA)
    omega_init_batch = _constant_init(n_file, INIT_OMEGA)
    r_init_batch = _constant_init(n_file, INIT_R)
    B_init_batch = _constant_init(n_file, INIT_B)

    file_label = _format_file_label(file_id)
    title_suffix = f"(File {file_label})"
    pdf_suffix = f"(File {file_label})"
    title_alpha_init = r"$\alpha_{\mathrm{init}}$"
    title_beta_init = r"$\beta_{\mathrm{init}}$"
    title_omega_init = r"$\omega_{\mathrm{init}}$"
    title_r_init = r"$\gamma_{\mathrm{init}}$"
    title_B_init = r"$B_{\mathrm{init}}$"

    plot_coefficient_histogram(alpha_batch, cat_tags_batch, fr"$\alpha$ {title_suffix}", init_values=alpha_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_alpha_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(alpha_init_batch, cat_tags_batch, f"{title_alpha_init} {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_alpha_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(beta_batch, cat_tags_batch, fr"$\beta$ {title_suffix}", init_values=beta_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_beta_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(beta_init_batch, cat_tags_batch, f"{title_beta_init} {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_beta_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(omega_batch, cat_tags_batch, fr"$\omega$ {title_suffix}", init_values=omega_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_omega_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(omega_init_batch, cat_tags_batch, f"{title_omega_init} {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_omega_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(r_batch, cat_tags_batch, fr"$\gamma$ {title_suffix}", bins=30, init_values=r_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_r_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(r_init_batch, cat_tags_batch, f"{title_r_init} {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_r_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(B_batch, cat_tags_batch, fr"$B$ {title_suffix}", init_values=B_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_B_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(B_init_batch, cat_tags_batch, f"{title_B_init} {title_suffix}", pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_B_init_file_{pdf_suffix}.pdf"))

    cat_tags_binary_batch = make_binary_cat_tags(cat_tags_batch)
    plot_coefficient_histogram(alpha_batch, cat_tags_binary_batch, fr"$\alpha$ {title_suffix}", colors=binary_histogram_colors, init_values=alpha_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_alpha_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(alpha_init_batch, cat_tags_binary_batch, f"{title_alpha_init} {title_suffix}", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_alpha_init_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(beta_batch, cat_tags_binary_batch, fr"$\beta$ {title_suffix}", colors=binary_histogram_colors, init_values=beta_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_beta_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(beta_init_batch, cat_tags_binary_batch, f"{title_beta_init} {title_suffix}", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_beta_init_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(omega_batch, cat_tags_binary_batch, fr"$\omega$ {title_suffix}", colors=binary_histogram_colors, init_values=omega_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_omega_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(omega_init_batch, cat_tags_binary_batch, f"{title_omega_init} {title_suffix}", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_omega_init_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(r_batch, cat_tags_binary_batch, fr"$\gamma$ {title_suffix}", colors=binary_histogram_colors, bins=30, init_values=r_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_r_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(r_init_batch, cat_tags_binary_batch, f"{title_r_init} {title_suffix}", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_r_init_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(B_batch, cat_tags_binary_batch, fr"$B$ {title_suffix}", colors=binary_histogram_colors, init_values=B_init_batch, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_B_binary_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(B_init_batch, cat_tags_binary_batch, f"{title_B_init} {title_suffix}", colors=binary_histogram_colors, pdf_filename=os.path.join(scatter_histogram_dir, f"histogram_B_init_binary_file_{pdf_suffix}.pdf"))

print(f"[DONE] Saved histogram PDFs from saved CSV -> {scatter_histogram_dir}")


**Evaluation Metric**

In [ ]:
# Evaluation metrics from saved parameter CSV (no retraining required)

import os
from pathlib import Path

import numpy as np
import pandas as pd

if "_load_params_df_for_plots" not in globals():
    def _load_params_df_for_plots():
        csv_candidates = [Path(WINDOW_PARAMS_CSV_PATH), Path(PARAM_CSV_PATH)]
        params_csv_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])
        if not params_csv_path.exists():
            raise FileNotFoundError(
                "Saved parameter CSV was not found. Expected one of: "
                + ", ".join(str(path) for path in csv_candidates)
            )

        params_df = pd.read_csv(params_csv_path)
        if "category" not in params_df.columns and "class" in params_df.columns:
            params_df = params_df.rename(columns={"class": "category"})

        required_cols = {"category", "file_idx", "window", "alpha", "beta", "omega", "r", "B"}
        missing_cols = required_cols - set(params_df.columns)
        if missing_cols:
            raise ValueError(f"CSV is missing required columns: {sorted(missing_cols)}")

        params_df = params_df.copy()
        params_df["category"] = params_df["category"].astype(str).replace(
            {
                "Health": "Healthy",
                "health": "Healthy",
                "wheeze": "Wheeze",
                "crackle": "Crackle",
            }
        )
        params_df["window"] = pd.to_numeric(params_df["window"], errors="coerce")
        params_df["file_idx"] = pd.to_numeric(params_df["file_idx"], errors="coerce")
        for col in ["alpha", "beta", "omega", "r", "B"]:
            params_df[col] = pd.to_numeric(params_df[col], errors="coerce")

        params_df = params_df.dropna(subset=["category", "file_idx", "window", "alpha", "beta", "omega", "r", "B"])
        params_df = params_df.sort_values(["file_idx", "category", "window"]).reset_index(drop=True)

        known_categories = {"Healthy", "Wheeze", "Crackle"}
        unknown_categories = sorted(set(params_df["category"]) - known_categories)
        if unknown_categories:
            raise ValueError(f"CSV contains categories without plot colors: {unknown_categories}")

        return params_df, params_csv_path

if "_format_file_label" not in globals():
    def _format_file_label(file_id):
        try:
            as_float = float(file_id)
            if as_float.is_integer():
                return str(int(as_float))
        except (TypeError, ValueError):
            pass
        return str(file_id)

if "_file_sort_key" not in globals():
    def _file_sort_key(file_id):
        label = _format_file_label(file_id)
        try:
            return (0, int(label))
        except ValueError:
            return (1, label)

if "_complete_file_ids_for_all" not in globals():
    def _complete_file_ids_for_all(params_df):
        expected_categories = [
            cat for cat in ["Healthy", "Wheeze", "Crackle"]
            if cat in set(params_df["category"])
        ]
        window_counts = (
            params_df
            .groupby(["file_idx", "category"])["window"]
            .nunique()
            .unstack(fill_value=0)
        )
        expected_window_counts = window_counts.reindex(columns=expected_categories, fill_value=0)
        return expected_window_counts.index[
            (expected_window_counts > 0).all(axis=1)
            & (expected_window_counts.nunique(axis=1) == 1)
        ].tolist()

import contextlib
import io
import sys
from scipy.stats import chi2, f_oneway
from sklearn.metrics import calinski_harabasz_score

params_df, params_csv_path_for_metrics = _load_params_df_for_plots()

expected_categories_for_global = [
    cat for cat in ["Healthy", "Wheeze", "Crackle"]
    if cat in set(params_df["category"])
]
window_counts_by_file = (
    params_df
    .groupby(["file_idx", "category"])["window"]
    .nunique()
    .unstack(fill_value=0)
)
expected_window_counts = window_counts_by_file.reindex(
    columns=expected_categories_for_global,
    fill_value=0,
)
complete_file_ids_for_global = expected_window_counts.index[
    (expected_window_counts > 0).all(axis=1)
    & (expected_window_counts.nunique(axis=1) == 1)
].tolist()
all_file_ids_for_info = sorted(params_df["file_idx"].unique().tolist())
excluded_file_ids_for_global = [
    file_id for file_id in all_file_ids_for_info
    if file_id not in set(complete_file_ids_for_global)
]
global_mask = params_df["file_idx"].isin(complete_file_ids_for_global).to_numpy()
if not global_mask.any():
    raise ValueError("No complete matched file sets are available for global evaluation metrics.")

labels_arr = params_df["category"].to_numpy()
file_ids = params_df["file_idx"].to_numpy()
cluster_ids = np.array([f"{lab}_{fid}" for lab, fid in zip(labels_arr, file_ids)])
n = len(params_df)
n_global = int(global_mask.sum())

alpha0 = 0.5
r0 = 1.0
omega0 = 2.0 * np.pi
beta0 = omega0**2
B0 = 0.5


def fisher_discriminant_ratio(vals, labels):
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    classes = np.unique(labels)
    means = [vals[labels == c].mean() for c in classes]
    variances = [vals[labels == c].var(ddof=1) for c in classes]
    between = np.var(means, ddof=1)
    within = np.mean(variances)
    return between / within if within > 0 else np.nan


def anova_F(vals, labels):
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    groups = [vals[labels == c] for c in np.unique(labels)]
    if any(len(g) < 2 for g in groups):
        return np.nan
    if all(np.all(g == g[0]) for g in groups):
        return np.nan
    return f_oneway(*groups).statistic


def anova_p_value(vals, labels):
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    groups = [vals[labels == c] for c in np.unique(labels)]
    if any(len(g) < 2 for g in groups):
        return np.nan
    if all(np.all(g == g[0]) for g in groups):
        return np.nan
    return f_oneway(*groups).pvalue


def ch_index(vals, labels):
    vals = np.asarray(vals, dtype=float).reshape(-1, 1)
    if np.allclose(vals, vals[0]):
        return np.nan
    return calinski_harabasz_score(vals, labels)


def cluster_robust_wald(vals, labels, clusters):
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    clusters = np.asarray(clusters)

    valid = np.isfinite(vals)
    vals = vals[valid]
    labels = labels[valid]
    clusters = clusters[valid]

    classes = np.unique(labels)
    unique_clusters = np.unique(clusters)
    if len(classes) < 2 or len(unique_clusters) < 3:
        return np.nan, np.nan
    if np.allclose(vals, vals[0]):
        return np.nan, np.nan

    n_obs = len(vals)
    X = np.ones((n_obs, len(classes)))
    for j, cls in enumerate(classes[1:], start=1):
        X[:, j] = (labels == cls).astype(float)

    p = X.shape[1]
    q = p - 1
    if n_obs <= p or np.linalg.matrix_rank(X) < p or len(unique_clusters) <= q:
        return np.nan, np.nan

    xtx_inv = np.linalg.pinv(X.T @ X)
    beta_hat = xtx_inv @ X.T @ vals
    resid = vals - X @ beta_hat

    meat = np.zeros((p, p))
    for g in unique_clusters:
        msk = clusters == g
        score_g = X[msk].T @ resid[msk]
        meat += np.outer(score_g, score_g)

    cov = xtx_inv @ meat @ xtx_inv
    cov *= (len(unique_clusters) / (len(unique_clusters) - 1)) * ((n_obs - 1) / (n_obs - p))

    R = np.zeros((q, p))
    R[:, 1:] = np.eye(q)
    diff = R @ beta_hat
    cov_restricted = R @ cov @ R.T

    if not np.all(np.isfinite(cov_restricted)) or np.linalg.matrix_rank(cov_restricted) < q:
        return np.nan, np.nan

    wald_stat = float(diff.T @ np.linalg.pinv(cov_restricted) @ diff)
    if wald_stat < 0 and np.isclose(wald_stat, 0):
        wald_stat = 0.0
    p_value = float(chi2.sf(wald_stat, q))
    return wald_stat, p_value


metrics = {
    "FDR": fisher_discriminant_ratio,
    "ANOVA-F": anova_F,
    "ANOVA-p": anova_p_value,
    "CH": ch_index,
}


def fmt(x):
    return f"{0.0:8.3f}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:8.3f}"


def fmt_p(name, x):
    if name == "ANOVA-p":
        return f"{0.0:10.3e}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:10.3e}"
    return fmt(x)


def fmt_wald(x):
    return f"{'nan':>8}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:8.3f}"


def print_block(title, init_vals, train_vals, labels, file_ids=None, cluster_ids=None, global_mask=None):
    init_vals = np.asarray(init_vals)
    train_vals = np.asarray(train_vals)
    labels = np.asarray(labels)

    if global_mask is None:
        global_mask = np.ones(len(train_vals), dtype=bool)
    else:
        global_mask = np.asarray(global_mask, dtype=bool)

    init_vals_global = init_vals[global_mask]
    train_vals_global = train_vals[global_mask]
    labels_global = labels[global_mask]

    print(f"\n\n==========  {title} Separation Metrics  ==========")
    print("-- Global --")
    print(f"{'Metric':<8} | {'Init':>10} | {'Trained':>10}")
    print("-" * 38)
    for name, fn in metrics.items():
        g0 = fn(init_vals_global, labels_global)
        g1 = fn(train_vals_global, labels_global)
        print(f"{name:<8} | {fmt_p(name, g0)} | {fmt_p(name, g1)}")

    if cluster_ids is not None and len(cluster_ids) == len(train_vals):
        cluster_ids_global = np.asarray(cluster_ids)[global_mask]
        w0, p0 = cluster_robust_wald(init_vals_global, labels_global, cluster_ids_global)
        w1, p1 = cluster_robust_wald(train_vals_global, labels_global, cluster_ids_global)
        print(f"{'CR-Wald':<10} | {fmt_wald(w0)} | {fmt_wald(w1)}")
        print(f"{'CR-Wald-p':<10} | {fmt_wald(p0)} | {fmt_wald(p1)}")

    if file_ids is not None and len(file_ids) == len(train_vals):
        unique_files = np.unique(file_ids)
        valid_files = [
            f for f in unique_files
            if len(np.unique(labels[file_ids == f])) >= 2
        ]

        if len(valid_files) == 0:
            print("\n[Per-file evaluation skipped: no file contains multiple classes]")
            return

        for f in valid_files:
            msk = file_ids == f
            if len(np.unique(labels[msk])) < 2:
                continue

            print(f"\n-- File {_format_file_label(f)} --")
            print(f"{'Metric':<10} | {'Init':>10} | {'Trained':>10}")
            print("-" * 40)
            for name, fn in metrics.items():
                v0 = fn(init_vals[msk], labels[msk])
                v1 = fn(train_vals[msk], labels[msk])
                print(f"{name:<10} | {fmt_p(name, v0)} | {fmt_p(name, v1)}")


def make_binary_labels(labels):
    return np.array([
        "Pneumonia" if label in ("Wheeze", "Crackle") else ("Healthy" if label == "Health" else label)
        for label in labels
    ])


class _EvalMetricTee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, text):
        for stream in self.streams:
            stream.write(text)
        return len(text)

    def flush(self):
        for stream in self.streams:
            stream.flush()


eval_metric_txt_path = Path(EVAL_METRIC_TXT_PATH)
eval_metric_txt_path.parent.mkdir(parents=True, exist_ok=True)
eval_metric_buffer = io.StringIO()

with contextlib.redirect_stdout(_EvalMetricTee(sys.stdout, eval_metric_buffer)):
    print(f"{RUN_NAME}\n")
    print("Evaluation Metric\n")
    print(f"[LOAD] {params_csv_path_for_metrics}")
    print(f"[INFO] n={n}, categories={params_df['category'].value_counts().to_dict()}, files={[_format_file_label(file_id) for file_id in all_file_ids_for_info]}")
    print("[GLOBAL INFO] window counts by file/category:")
    print(expected_window_counts.to_string())
    print(f"[GLOBAL INFO] used file_idx for Global: {[_format_file_label(file_id) for file_id in sorted(complete_file_ids_for_global)]}")
    print(f"[GLOBAL INFO] excluded file_idx from Global: {[_format_file_label(file_id) for file_id in excluded_file_ids_for_global]}")
    print(f"[GLOBAL INFO] n_global={n_global}")

    alpha_init_vals = np.full(n, alpha0)
    beta_init_vals = np.full(n, beta0)
    r_init_vals = np.full(n, r0)
    B_init_vals = np.full(n, B0)
    omega_init_vals = np.full(n, omega0)

    alpha_train_vals = params_df["alpha"].to_numpy()
    beta_train_vals = params_df["beta"].to_numpy()
    r_train_vals = params_df["r"].to_numpy()
    B_train_vals = params_df["B"].to_numpy()
    omega_train_vals = params_df["omega"].to_numpy()

    print_block("Alpha", alpha_init_vals, alpha_train_vals, labels_arr, file_ids, cluster_ids, global_mask)
    print_block("Beta", beta_init_vals, beta_train_vals, labels_arr, file_ids, cluster_ids, global_mask)
    print_block("Gamma", r_init_vals, r_train_vals, labels_arr, file_ids, cluster_ids, global_mask)
    print_block("B", B_init_vals, B_train_vals, labels_arr, file_ids, cluster_ids, global_mask)
    print_block("Omega", omega_init_vals, omega_train_vals, labels_arr, file_ids, cluster_ids, global_mask)

    labels_binary_arr = make_binary_labels(labels_arr)
    cluster_binary_ids = np.array([f"{lab}_{fid}" for lab, fid in zip(labels_binary_arr, file_ids)])
    print("\n\n==========  Healthy vs Pneumonia Binary Classes  ==========")
    print_block("Alpha (Binary)", alpha_init_vals, alpha_train_vals, labels_binary_arr, file_ids, cluster_binary_ids, global_mask)
    print_block("Beta (Binary)", beta_init_vals, beta_train_vals, labels_binary_arr, file_ids, cluster_binary_ids, global_mask)
    print_block("Gamma (Binary)", r_init_vals, r_train_vals, labels_binary_arr, file_ids, cluster_binary_ids, global_mask)
    print_block("B (Binary)", B_init_vals, B_train_vals, labels_binary_arr, file_ids, cluster_binary_ids, global_mask)
    print_block("Omega (Binary)", omega_init_vals, omega_train_vals, labels_binary_arr, file_ids, cluster_binary_ids, global_mask)

eval_metric_txt_path.write_text(eval_metric_buffer.getvalue(), encoding="utf-8")
print(f"[SAVE] Evaluation Metric txt -> {eval_metric_txt_path}")


**Saved Model Load**

In [ ]:
def _parse_model_filename(fn: str):

    if not (fn.startswith("model_") and fn.endswith(".pt")):
        return None, None, None

    name = fn[len("model_"):-len(".pt")]  

    if "_window_" not in name:
        return None, None, None

    left, win_str = name.rsplit("_window_", 1)

    try:
        win = int(win_str)
    except ValueError:
        return None, None, None

    # left = "{file_idx}_{category}"
    if "_" not in left:
        return None, None, None

    file_idx, cat = left.rsplit("_", 1)

    return file_idx, cat, win


def export_pinn_params_to_csv(
    models_dir=SAVED_MODEL_DIR,
    csv_path=PARAM_CSV_PATH,
    ModelClass=MlpModel,   
    cfg=None,
    device="cpu",
    win_sec=None,      
):

    if ModelClass is None or cfg is None:
        raise ValueError("Both ModelClass and cfg must be provided.")
    if win_sec is None:
        raise ValueError("win_sec must be specified (e.g., win_sec=WIN_SEC).")

    omega_init = 2.0 * np.pi
    beta_init  = omega_init**2
    B_init     = 0.5
    scaling_factor = 1.0 / win_sec

    alpha_list = []
    beta_list  = []
    omega_list = []
    r_list     = []
    B_list     = []
    phi_list   = [] 
    cat_tags   = []
    win_ids    = []  

    alpha_hat_list = []
    beta_hat_list  = []
    omega_hat_list = []
    r_hat_list     = []
    B_hat_list     = []

    rows = []
    selection_map = {}
    selection_dir = os.path.join(models_dir, "selected_windows")
    if os.path.isdir(selection_dir):
        for selection_csv in glob.glob(os.path.join(selection_dir, "selected_windows_file_*.csv")):
            df_sel = pd.read_csv(selection_csv)
            required_sel_cols = {"file_idx", "category", "window", "train_order", "random_select_order"}
            if not required_sel_cols.issubset(df_sel.columns):
                continue
            for _, row_sel in df_sel.iterrows():
                key = (str(row_sel["file_idx"]), str(row_sel["category"]), int(row_sel["window"]))
                selection_map[key] = {
                    "train_order": int(row_sel["train_order"]),
                    "random_select_order": int(row_sel["random_select_order"]),
                }

    fns = sorted(
    [fn for fn in os.listdir(models_dir) if fn.endswith(".pt") and fn.startswith("model_")],
    key=lambda fn: (
        _parse_model_filename(fn)[0],  # file_idx
        _parse_model_filename(fn)[1],  # category
        _parse_model_filename(fn)[2],  # window
        )
    )   

    for fn in fns:
        file_idx, cat, win = _parse_model_filename(fn)
        if file_idx is None:
            continue

        selection_key = (str(file_idx), str(cat), int(win))
        if selection_map and selection_key not in selection_map:
            continue
        selection_info = selection_map.get(selection_key, {})

        model_path = os.path.join(models_dir, fn)

        # Recreate the model using the same architecture as training.
        model = ModelClass(
                    in_dim=1,
                    n_hidden=cfg.n_hidden,
                    n_neurons=cfg.n_neurons,
                    beta_init=beta_init, 
                    omega_init=omega_init, 
                    B_init=B_init, 
                    fourier_m=cfg.fourier_m,
                    fourier_sigma=cfg.fourier_sigma,
                    use_fourier=cfg.use_fourier,
                    scaling_factor=scaling_factor, # Must match the value used during training.
                ).to(device)

        sd = torch.load(model_path, map_location=device)
        model.load_state_dict(sd)
        model.eval()

        # Extract parameters in physical units.
        alpha = float(model.alpha.item())
        beta  = float(model.beta.item())
        omega = float(model.omega.item())
        r     = float(model.r.item())
        B     = float(model.B.item())
        phi   = float(model.phi.item())

        # Extract parameters in the scaled domain.
        alpha_hat = float(model.alpha_hat.item())
        beta_hat  = float(model.beta_hat.item())
        omega_hat = float(model.omega_hat.item())
        r_hat     = float(model.r_hat.item())
        B_hat     = float(model.B_hat.item())

        alpha_list.append(alpha)
        beta_list.append(beta)
        omega_list.append(omega)
        r_list.append(r)
        B_list.append(B)
        phi_list.append(phi)
        cat_tags.append(cat)
        win_ids.append(win)

        alpha_hat_list.append(alpha_hat)
        beta_hat_list.append(beta_hat)
        omega_hat_list.append(omega_hat)
        r_hat_list.append(r_hat)
        B_hat_list.append(B_hat)

        rows.append({
            "model_file": fn,
            "model_path": model_path,
            "file_idx": file_idx,
            "category": cat,
            "window": win,
            "train_order": selection_info.get("train_order"),
            "random_select_order": selection_info.get("random_select_order"),
            "win_sec": float(win_sec),
            "scaling_factor": float(scaling_factor),

            # physical units
            "alpha": alpha,
            "beta": beta,
            "omega": omega,
            "r": r,
            "B": B,
            "phi": phi,
            
            # scaled domain
            "alpha_hat": alpha_hat,
            "beta_hat": beta_hat,
            "omega_hat": omega_hat,
            "r_hat": r_hat,
            "B_hat": B_hat,
        })

        del model
        torch.cuda.empty_cache()

    # Save results to a CSV file using the standard csv module.
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model_file","model_path","file_idx","category","window",
                "train_order","random_select_order",
                "win_sec","scaling_factor","alpha","beta","omega","r","B","phi",
                "alpha_hat","beta_hat","omega_hat","r_hat","B_hat",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"[DONE] Saved CSV -> {csv_path} (n_models={len(rows)})")

    return alpha_list, beta_list, omega_list, r_list, B_list, phi_list, alpha_hat_list, beta_hat_list, omega_hat_list, r_hat_list, B_hat_list, cat_tags, win_ids, rows


# call
alpha_list, beta_list, omega_list, r_list, B_list, phi_list, \
alpha_hat_list, beta_hat_list, omega_hat_list, r_hat_list, B_hat_list, \
cat_tags, win_ids, rows = export_pinn_params_to_csv(
    models_dir=SAVED_MODEL_DIR,
    csv_path=PARAM_CSV_PATH,
    ModelClass=MlpModel,
    cfg=cfg,
    device="cpu",
    win_sec=WIN_SEC
)

**GT vs PINN vs ODE Forward**

In [ ]:
models_dir = Path(SAVED_MODEL_DIR)
params_csv_path = Path(PARAM_CSV_PATH)
pdf_dir = Path(globals().get('ODE_PLOT_DIR', PDF_DIR / 'ODE_plot'))
pdf_dir.mkdir(parents=True, exist_ok=True)

WIN_SEC_LOCAL = globals().get('WIN_SEC', 0.1)
scaling_factor = 1.0 / WIN_SEC_LOCAL
n_plot = 10
plot_each_window = True
plot_window_chunks = True
plot_full_signal = True

device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

required_window_vars = ['window_H', 'window_W', 'window_C']
missing_window_vars = [name for name in required_window_vars if name not in globals()]
if missing_window_vars:
    raise NameError(
        'Run the data/window preprocessing cells first. Missing: '
        + ', '.join(missing_window_vars)
    )

required_cols = {
    'file_idx', 'category', 'window', 'model_path',
    'alpha_hat', 'beta_hat', 'omega_hat', 'r_hat', 'B_hat', 'phi'
}


def rebuild_param_csv_from_saved_models():
    if 'export_pinn_params_to_csv' not in globals():
        raise RuntimeError(
            'Parameter CSV is missing/incomplete and export_pinn_params_to_csv() is not defined. '
            'Run the Saved Model Load/export cell first, then rerun this plot cell.'
        )

    return export_pinn_params_to_csv(
        models_dir=str(models_dir),
        csv_path=str(params_csv_path),
        ModelClass=MlpModel,
        cfg=cfg,
        device='cpu',
        win_sec=WIN_SEC_LOCAL,
    )


if not params_csv_path.exists():
    print(f'[INFO] CSV not found. Rebuilding from saved models: {params_csv_path}')
    rebuild_param_csv_from_saved_models()

df_params = pd.read_csv(params_csv_path)
missing_cols = required_cols - set(df_params.columns)
if missing_cols:
    print(f'[INFO] CSV is missing {sorted(missing_cols)}. Rebuilding from saved models.')
    rebuild_param_csv_from_saved_models()
    df_params = pd.read_csv(params_csv_path)

df_params['file_idx'] = df_params['file_idx'].astype(int)
df_params['window'] = df_params['window'].astype(int)


def relu_no_utt_forward_saved(t, u0, alpha, beta, r, B, omega, phi=0.0,
                              method='RK45', rtol=1e-7, atol=1e-9):
    """
    ReLU-degenerate forward model without u'':
        alpha*u_tau + beta*u + r*u^3 = B*cos(omega*tau + phi)

    This matches the ODE forward method used in the markdown-adjacent
    GT vs ODE (w/o u'') cell. Only u0 is used as the initial condition.
    """
    t = np.asarray(t, dtype=np.float64).ravel()
    alpha = float(alpha)
    beta = float(beta)
    r = float(r)
    B = float(B)
    omega = float(omega)
    phi = float(phi)

    if np.isclose(alpha, 0.0):
        raise RuntimeError("alpha is too close to zero for ReLU no-u'' first-order forward")

    def rhs(ti, y):
        u = y[0]
        du = (B * np.cos(omega * ti + phi) - beta * u - r * u**3) / alpha
        return [du]

    sol = solve_ivp(
        rhs,
        t_span=(float(t[0]), float(t[-1])),
        y0=[float(u0)],
        t_eval=t,
        method=method,
        rtol=rtol,
        atol=atol,
    )

    if not sol.success:
        raise RuntimeError(f"ReLU no-u'' ODE solve failed: {sol.message}")

    return sol.y[0]

def fd_initial_conditions_saved(t_raw, x_raw):
    t_raw = np.asarray(t_raw, dtype=np.float64).ravel()
    x_raw = np.asarray(x_raw, dtype=np.float64).ravel()
    dt = float(np.diff(t_raw).mean())
    u0 = float(x_raw[0])

    if len(x_raw) >= 3:
        v0_dt = (-3 * x_raw[0] + 4 * x_raw[1] - x_raw[2]) / (2 * dt)
    else:
        v0_dt = (x_raw[1] - x_raw[0]) / dt

    return u0, float(v0_dt / scaling_factor)  # du/dtau


def make_model_for_saved_state():
    omega_init = 2.0 * np.pi
    beta_init = omega_init**2
    B_init = 0.5

    return MlpModel(
        in_dim=1,
        n_hidden=cfg.n_hidden,
        n_neurons=cfg.n_neurons,
        beta_init=beta_init,
        omega_init=omega_init,
        B_init=B_init,
        fourier_m=cfg.fourier_m,
        fourier_sigma=cfg.fourier_sigma,
        use_fourier=cfg.use_fourier,
        scaling_factor=scaling_factor,
    )


def load_saved_model(model_path):
    model = make_model_for_saved_state().to(device)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def get_param_row(file_idx, category_name, window):
    row = df_params[
        (df_params['file_idx'] == int(file_idx)) &
        (df_params['category'] == category_name) &
        (df_params['window'] == int(window))
    ]
    return None if row.empty else row.iloc[0]


def model_path_from_row(row, file_idx, category_name, window):
    if row is not None and 'model_file' in row.index and pd.notna(row['model_file']):
        return models_dir / str(row['model_file'])
    if row is not None and 'model_path' in row.index and pd.notna(row['model_path']):
        model_path = Path(row['model_path'])
        return model_path if not model_path.is_absolute() else models_dir / model_path.name
    return models_dir / f'model_{file_idx}_{category_name}_window_{window}.pt'


def save_current_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if 'save_plot_as_pdf' in globals():
        save_plot_as_pdf(str(path))
    else:
        plt.savefig(path, format='pdf', bbox_inches='tight')


def apply_train_prediction_plot_format(title_msg):
    """Match the prediction plot format used inside train_all_windows()."""
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Amplitude', fontsize=14)
    plt.title(title_msg, fontsize=14)
    plt.legend(fontsize=12)
    plt.tick_params(axis='both', which='major', labelsize=14)
    ax = plt.gca()
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    plt.tight_layout()


file_windows_by_category = {
    'Healthy': window_H,
    'Wheeze': window_W,
    'Crackle': window_C,
}

ic_rows = []
missing_param_rows = []
missing_model_paths = []

print(f'[CSV LOAD] {params_csv_path}')
print(f'[CSV ROWS] {len(df_params)} parameter rows')
print(f'[MODEL DIR] {models_dir}')
print(f'[SAVE DIR] {pdf_dir}')
print("[ODE MODEL] ReLU no-u'': alpha*u_tau + beta*u + r*u^3 = B*cos(omega*tau + phi)")

for category_name, file_windows in file_windows_by_category.items():
    for file_idx, file_data in enumerate(file_windows, start=1):
        print(f'\n[FILE] {category_name} file={file_idx}')

        t_file_all, gt_file_all, pinn_file_all, ode_file_all = [], [], [], []

        t_windows, x_windows = file_data[:2]
        x_windows_raw = np.asarray(x_windows) 

        for win_zero_based in range(len(t_windows)):
            window = win_zero_based + 1
            row = get_param_row(file_idx, category_name, window)
            if row is None:
                missing_param_rows.append((category_name, file_idx, window))
                continue

            model_path = model_path_from_row(row, file_idx, category_name, window)
            if not model_path.exists():
                missing_model_paths.append(str(model_path))
                continue

            t_raw = np.asarray(t_windows[win_zero_based], dtype=np.float64).ravel()
            x_raw = np.asarray(x_windows_raw[win_zero_based], dtype=np.float64).ravel()
            tau_local = scaling_factor * t_raw
            t_plot = t_raw + win_zero_based * WIN_SEC_LOCAL

            model = load_saved_model(model_path)
            with torch.no_grad():
                tau_tensor = torch.as_tensor(
                    tau_local[:, None],
                    dtype=torch.float32,
                    device=device,
                )
                u_pinn = model(tau_tensor).detach().cpu().numpy().squeeze()

                # Use the parameters from the loaded model so PINN and ODE always match the same checkpoint.
                alpha_hat = float(model.alpha_hat.item())
                beta_hat = float(model.beta_hat.item())
                omega_hat = float(model.omega_hat.item())
                r_hat = float(model.r_hat.item())
                B_hat = float(model.B_hat.item())
                phi = float(model.phi.item())

            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            u0, v0_tau = fd_initial_conditions_saved(t_raw, x_raw)  # v0_tau is diagnostics only; no-u'' ODE uses u0.

            try:
                u_ode = relu_no_utt_forward_saved(
                    t=tau_local,
                    u0=u0,
                    alpha=alpha_hat,
                    beta=beta_hat,
                    r=r_hat,
                    B=B_hat,
                    omega=omega_hat,
                    phi=phi,
                )
                ode_success = True
                ode_fail_msg = ''
            except RuntimeError as e:
                u_ode = np.full_like(x_raw, np.nan)
                ode_success = False
                ode_fail_msg = str(e)
                print(f'[ODE FAIL] {category_name} file={file_idx} win={window} | {ode_fail_msg}')

            t_file_all.append(t_plot)
            gt_file_all.append(x_raw)
            pinn_file_all.append(u_pinn)
            ode_file_all.append(u_ode)

            if window <= n_plot:
                ic_rows.append({
                    'category': category_name,
                    'file_idx': file_idx,
                    'window': window,
                    'u0': u0,
                    'v0_tau': v0_tau,
                    'alpha_hat': alpha_hat,
                    'beta_hat': beta_hat,
                    'omega_hat': omega_hat,
                    'r_hat': r_hat,
                    'B_hat': B_hat,
                    'phi': phi,
                    'model_path': str(model_path),
                    'ode_success': ode_success,
                    'ode_fail_msg': ode_fail_msg,
                })

                print(
                    f'[IC] {category_name} | file={file_idx} | win={window} | '
                    f'u0={u0:.6e}, v0_tau={v0_tau:.6e}'
                )

                if plot_each_window:
                    plt.figure(figsize=(8, 3))
                    plt.plot(t_plot, u_pinn, label='PINN pred')
                    plt.plot(t_plot, x_raw, '--', label='Ground Truth')
                    plt.plot(t_plot, u_ode, color='green', label="ODE forward (ReLU w/o u'')")
                    title_msg = f"{category_name} window-{window} prediction | ODE (w/o u'')"
                    if not ode_success:
                        title_msg += ' | ODE FAIL'
                    apply_train_prediction_plot_format(title_msg)

                    pdf_filename = pdf_dir / f'{category_name}_file{file_idx}_win{window:03d}_GT_PINN_ODE.pdf'
                    save_current_fig(pdf_filename)
                    print(f'[SAVE] {pdf_filename}')
                    plt.show()
                    plt.close()

        if plot_window_chunks and t_file_all:
            n_windows_file = len(t_file_all)
            for chunk_start in range(0, n_windows_file, n_plot):
                chunk_end = min(chunk_start + n_plot, n_windows_file)
                first_window = chunk_start + 1
                last_window = chunk_end

                t_chunk = np.concatenate(t_file_all[chunk_start:chunk_end])
                gt_chunk = np.concatenate(gt_file_all[chunk_start:chunk_end])
                pinn_chunk = np.concatenate(pinn_file_all[chunk_start:chunk_end])
                ode_chunk = np.concatenate(ode_file_all[chunk_start:chunk_end])

                plt.figure(figsize=(8, 3))
                plt.plot(t_chunk, pinn_chunk, label='PINN pred')
                plt.plot(t_chunk, gt_chunk, '--', label='Ground Truth')
                plt.plot(t_chunk, ode_chunk, color='green', label="ODE forward (ReLU w/o u'')")
                apply_train_prediction_plot_format(
                    f"{category_name} windows {first_window}-{last_window} prediction | ODE (w/o u'')"
                )

                pdf_filename = pdf_dir / f'{category_name}_file{file_idx}_win{first_window:03d}-{last_window:03d}_GT_PINN_ODE.pdf'
                save_current_fig(pdf_filename)
                print(f'[SAVE FILE PLOT] {pdf_filename}')
                plt.show()
                plt.close()

        if plot_full_signal and t_file_all:
            t_all = np.concatenate(t_file_all)
            gt_all = np.concatenate(gt_file_all)
            pinn_all = np.concatenate(pinn_file_all)
            ode_all = np.concatenate(ode_file_all)

            plt.figure(figsize=(8, 3))
            plt.plot(t_all, pinn_all, label='PINN pred')
            plt.plot(t_all, gt_all, '--', label='Ground Truth')
            plt.plot(t_all, ode_all, color='green', label="ODE forward (ReLU w/o u'')")
            apply_train_prediction_plot_format(
                f"{category_name} full signal prediction | ODE (w/o u'')"
            )

            pdf_filename = pdf_dir / f'{category_name}_file{file_idx}_FULL_GT_PINN_ODE.pdf'
            save_current_fig(pdf_filename)
            print(f'[SAVE FILE PLOT] {pdf_filename}')
            plt.show()
            plt.close()

ic_csv_path = pdf_dir / 'GT_PINN_ODE_initial_conditions.csv'
if ic_rows:
    pd.DataFrame(ic_rows).to_csv(ic_csv_path, index=False)
    print(f'[DONE] Saved initial conditions CSV -> {ic_csv_path}')

if missing_param_rows:
    print(f'[WARN] Missing parameter rows: {len(missing_param_rows)}')
    print(missing_param_rows[:10])

if missing_model_paths:
    print(f'[WARN] Missing model files: {len(missing_model_paths)}')
    print(missing_model_paths[:10])

print("[DONE] GT vs PINN vs ODE forward plots generated.")